# 02. 전성분 정규화 및 표준 성분 매칭

**목적**  
상품별 전성분 문자열을 long 형식으로 변환하고 표준 성분명과 매칭합니다.

**입력**  
`data/raw/올리브영_크림240개_상품명_전성분.csv`

**출력**  
`data/interim/전성분_정규화_세로형.csv 및 매칭 로그`

> 기본값에서는 외부 MFDS API를 호출하지 않고 저장된 전처리 결과를 확인합니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 안전 기본값: 저장된 전처리 결과만 확인합니다.
RUN_PREPROCESSING = False
print("전처리 실행:", RUN_PREPROCESSING)


## 전처리 코드

MFDS API를 포함한 전체 전처리는 `RUN_PREPROCESSING=True`일 때만 실행됩니다.


# 크림 전성분 Raw → MFDS 표준화 Long 데이터 파이프라인 v3.2

이 노트북은 크롤링한 크림 상품의 원본 전성분을 **제품×성분 Long 데이터**로 변환합니다.

## v3.2 핵심 보완

- 실제 MFDS API 컬럼명(`INGR_KOR_NAME` 등)을 기본 연결
- 21,833건 원본 캐시는 재다운로드 없이 표준 사전으로 재구성
- MFDS 매칭률이 80% 미만이면 저장 전에 강제 중단
- 카밍패드·PDRN 앰플 등 구분자 없는 증정품 성분 제거
- 제품명 접두부가 첫 성분에 붙는 문제 제거
- 따옴표·마침표·자연유래 설명·문자 오염 정리
- 괄호 없는 함량(`병풀잎추출물0.2%`) 분리
- CI 번호·원료 별칭 괄호는 MFDS 표준명으로 안전하게 매핑
- 복합 구성품의 서로 다른 component 사이 중복 로그 제거
- 2종 택1 및 메인 제품을 확정할 수 없는 복수 크림 상품 제외

메인 출력은 원본, Long 변환본, 애매 로그 3개입니다.



## 셀 1. Google Drive 마운트

Colab에서 Google Drive의 입력 파일과 출력 폴더에 접근합니다.



## 셀 2. 사용자 설정

보통 아래 항목만 수정하면 됩니다.

- `PROJECT_DIR`: 입력 파일이 있는 **폴더**
- `INPUT_FILENAME`: 파일명만 입력
- `MFDS_API_KEY_DIRECT`: 비워두면 Colab Secret 또는 실행 중 입력
- `REFRESH_MFDS_CACHE`: 성분사전을 새로 내려받을 때만 `True`

### API 인증키

권장 순서:

1. Colab 왼쪽 메뉴의 **열쇠 아이콘(Secrets)** 에 `MFDS_API_KEY` 등록
2. 등록하지 않았다면 실행 중 숨김 입력창에 인증키 붙여넣기
3. 공공데이터포털의 Encoding 키를 넣어도 내부에서 Decoding 형태로 변환


In [ ]:
if RUN_PREPROCESSING:
    pass
    # 파일과 컬럼 설정
    INPUT_FILENAME = "올리브영_크림240개_상품명_전성분.csv"

    PRODUCT_ID_COL = "상품번호"
    PRODUCT_NAME_COL = "상품명"
    INGREDIENT_TEXT_COL = "주성분"

    # 식품의약품안전처 화장품 원료성분정보 API
    USE_MFDS_API = True
    MFDS_API_BASE_URL = (
        "https://apis.data.go.kr/1471000/"
        "CsmtcsIngdCpntInfoService01"
    )
    MFDS_OPERATION_PATH = "getCsmtcsIngdCpntInfoService01"
    MFDS_API_KEY_DIRECT = ""
    REFRESH_MFDS_CACHE = False
    MFDS_PAGE_SIZE = 100

    MFDS_FIELD_OVERRIDES = {
        "standard_name": "INGR_KOR_NAME",
        "english_name": "INGR_ENG_NAME",
        "cas_no": "CAS_NO",
        "alias": "INGR_SYNONYM",
        "origin_definition": "ORIGIN_MAJOR_KOR_NAME",
        "ingredient_code": None,
    }

    NAME_SIMILARITY_THRESHOLD = 0.82
    WHITESPACE_MIN_COVERAGE = 0.95
    MIN_MFDS_MATCH_RATE = 0.80
    KEEP_INFO_LOGS = True

    RAW_OUTPUT_FILENAME = "01_원본정리.csv"
    LONG_OUTPUT_FILENAME = "전성분_정규화_세로형.csv"
    LOG_OUTPUT_FILENAME = "03_전처리_검토로그.csv"
    CACHE_DIRNAME = "_cache"
    MFDS_RAW_CACHE_FILENAME = "mfds_api_raw_cache.csv"
    MFDS_MASTER_CACHE_FILENAME = "mfds_ingredient_master_cache.csv"
    MFDS_META_CACHE_FILENAME = "mfds_api_meta.json"



## 셀 3. 라이브러리 설치 및 공통 함수

- `.xls` Excel 바이너리 지원을 위해 `xlrd` 설치
- 확장자는 `.xls`지만 실제 내용이 CSV인 파일도 자동 판별
- 문자열·경로·해시 공통 함수를 정의


In [ ]:
if RUN_PREPROCESSING:
    pass

    from collections import defaultdict
    from dataclasses import dataclass
    from datetime import datetime
    from difflib import SequenceMatcher
    from pathlib import Path
    from typing import Any, Iterable
    from urllib.parse import unquote
    import hashlib
    import json
    import math
    import re
    import time
    import unicodedata
    import xml.etree.ElementTree as ET

    import numpy as np
    import pandas as pd
    import requests

    # -----------------------------------------------------------------------------
    # Generic normalization
    # -----------------------------------------------------------------------------

    def normalize_unicode(value: Any) -> str:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return ""
        return unicodedata.normalize("NFKC", str(value))


    def normalize_spaces(value: Any) -> str:
        text = normalize_unicode(value)
        text = text.replace("\u00a0", " ").replace("\u200b", "")
        return re.sub(r"[ \t\r\n]+", " ", text).strip()


    def comparison_key(value: Any) -> str:
        text = normalize_spaces(value).lower()
        text = text.replace("ㆍ", "·")
        return re.sub(r"\s+", "", text)


    def sha256_text(text: str) -> str:
        return hashlib.sha256(text.encode("utf-8")).hexdigest()


    def name_similarity(left: str, right: str) -> float:
        return SequenceMatcher(None, comparison_key(left), comparison_key(right)).ratio()


    # -----------------------------------------------------------------------------
    # File loading
    # -----------------------------------------------------------------------------


In [ ]:
if RUN_PREPROCESSING:
    def read_csv_with_fallback(path: Path) -> pd.DataFrame:
        errors = []
        for encoding in ("utf-8-sig", "utf-8", "cp949"):
            try:
                frame = pd.read_csv(path, encoding=encoding)
                print(f"CSV 인코딩: {encoding}")
                return frame
            except (UnicodeDecodeError, pd.errors.ParserError) as exc:
                errors.append(f"{encoding}: {exc}")
        raise ValueError("CSV를 읽지 못했습니다.\n" + "\n".join(errors))


    def looks_like_excel_binary(path: Path) -> bool:
        with path.open("rb") as file:
            head = file.read(8)
        return head.startswith(b"\xD0\xCF\x11\xE0") or head.startswith(b"PK\x03\x04")


    def read_table_with_fallback(path: Path) -> pd.DataFrame:
        suffix = path.suffix.lower()
        if suffix in {".xls", ".xlsx"} and looks_like_excel_binary(path):
            engine = "xlrd" if suffix == ".xls" else "openpyxl"
            try:
                print(f"Excel 바이너리 로딩: {suffix}, engine={engine}")
                return pd.read_excel(path, engine=engine)
            except Exception as exc:
                print(f"Excel 로딩 실패 후 CSV 방식 재시도: {exc}")
        return read_csv_with_fallback(path)



## 셀 4. 경로 생성과 원본 파일 로딩

입력 파일이 진짜 Excel인지, 확장자만 `.xls`인 CSV인지 자동으로 판별합니다.

원본에는 내부 추적용 `source_row_id`만 추가하고, 다른 값은 변경하지 않습니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    PROJECT_PATH = DATA_INTERIM_DIR
    PROJECT_PATH.mkdir(parents=True, exist_ok=True)

    INPUT_PATH = DATA_RAW_DIR / INPUT_FILENAME
    RAW_OUTPUT_PATH = PROJECT_PATH / RAW_OUTPUT_FILENAME
    LONG_OUTPUT_PATH = PROJECT_PATH / LONG_OUTPUT_FILENAME
    LOG_OUTPUT_PATH = PROJECT_PATH / LOG_OUTPUT_FILENAME

    CACHE_PATH = PROJECT_PATH / CACHE_DIRNAME
    CACHE_PATH.mkdir(parents=True, exist_ok=True)

    MFDS_RAW_CACHE_PATH = CACHE_PATH / MFDS_RAW_CACHE_FILENAME
    MFDS_MASTER_CACHE_PATH = CACHE_PATH / MFDS_MASTER_CACHE_FILENAME
    MFDS_META_CACHE_PATH = CACHE_PATH / MFDS_META_CACHE_FILENAME

    pd.set_option("display.max_columns", 100)
    pd.set_option("display.max_colwidth", 180)

    print("입력 파일:", INPUT_PATH)
    print("메인 출력 폴더:", PROJECT_PATH)
    print("API 캐시 폴더:", CACHE_PATH)

    raw_df = read_table_with_fallback(INPUT_PATH)
    raw_df = raw_df.loc[
        :, ~raw_df.columns.astype(str).str.startswith("Unnamed:")
    ].copy()

    required_columns = {
        PRODUCT_ID_COL,
        PRODUCT_NAME_COL,
        INGREDIENT_TEXT_COL,
    }
    missing_columns = required_columns - set(raw_df.columns)

    if missing_columns:
        raise KeyError(
            f"입력 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}"
        )

    if "source_row_id" in raw_df.columns:
        raw_df = raw_df.drop(columns="source_row_id")

    raw_df.insert(0, "source_row_id", range(len(raw_df)))
    raw_df.to_csv(
        RAW_OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"원본 행 수: {len(raw_df):,}")
    display(raw_df.head(3))



## 셀 5. MFDS API 호출 도우미

제공된 Base URL 뒤의 상세 기능 경로를 자동으로 확인합니다.

우선 확인하는 대표 경로:

- `getCsmtcsIngdCpntInfoList01`
- `getCsmtcsIngdCpntInfoList`
- 기타 유사 경로

응답은 JSON과 XML을 모두 처리합니다.  
Swagger에서 상세 기능명이 확인되면 설정 셀의 `MFDS_OPERATION_PATH`에 직접 입력하는 것이 가장 확실합니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    # -----------------------------------------------------------------------------
    # MFDS API client v3.1
    #
    # 핵심 수정:
    # - HTTP 200 또는 resultCode=00만으로 엔드포인트를 확정하지 않습니다.
    # - 실제 성분 item이 1개 이상 반환될 때만 올바른 엔드포인트로 확정합니다.
    # - JSON 강제 파라미터보다 기본 XML 응답도 먼저 확인합니다.
    # - 엔드포인트/응답형식/페이지 파라미터 조합을 모두 시험합니다.
    # - 실패 시 응답 코드·메시지·payload 앞부분을 출력합니다.
    # -----------------------------------------------------------------------------

    DEFAULT_MFDS_OPERATION_CANDIDATES = [
        "getCsmtcsIngdCpntInfoList01",
        "getCsmtcsIngdCpntInfoList",
        "getCsmtcsIngdCpntInfo01",
        "getCsmtcsIngdCpntInfo",
        "getCsmtcsIngdCpntInfoService01",
    ]

    # 기본 XML을 먼저 확인합니다.
    DEFAULT_RESPONSE_TYPE_PARAM_CANDIDATES = [
        {},
        {"type": "json"},
        {"_type": "json"},
        {"returnType": "json"},
    ]

    # 공공데이터 API마다 페이지 변수명이 다른 경우에 대비합니다.
    DEFAULT_PAGING_PARAM_CANDIDATES = [
        ("pageNo", "numOfRows"),
        ("page", "perPage"),
        ("pageNo", "rows"),
    ]


    def _xml_element_to_obj(element: ET.Element) -> Any:
        children = list(element)
        if not children:
            return (element.text or "").strip()

        result: dict[str, Any] = {}

        for child in children:
            value = _xml_element_to_obj(child)
            key = child.tag.split("}")[-1]

            if key in result:
                if not isinstance(result[key], list):
                    result[key] = [result[key]]
                result[key].append(value)
            else:
                result[key] = value

        return result


In [ ]:
if RUN_PREPROCESSING:
    def parse_api_payload(response: requests.Response) -> Any:
        content_type = response.headers.get("content-type", "").lower()
        text = response.text.strip()

        if not text:
            raise ValueError("API 응답 본문이 비어 있습니다.")

        if (
            "json" in content_type
            or text.startswith("{")
            or text.startswith("[")
        ):
            return response.json()

        if text.startswith("<"):
            root = ET.fromstring(text)
            return {
                root.tag.split("}")[-1]:
                _xml_element_to_obj(root)
            }

        raise ValueError(
            "JSON/XML이 아닌 응답입니다: "
            f"{text[:500]}"
        )


    def recursive_find_lists(
        obj: Any,
        path: str = "root",
    ) -> list[tuple[str, list]]:
        found = []

        if isinstance(obj, list):
            if obj and all(
                isinstance(item, dict)
                for item in obj
            ):
                found.append((path, obj))

            for index, item in enumerate(obj):
                found.extend(
                    recursive_find_lists(
                        item,
                        f"{path}[{index}]",
                    )
                )

        elif isinstance(obj, dict):
            for key, value in obj.items():
                found.extend(
                    recursive_find_lists(
                        value,
                        f"{path}.{key}",
                    )
                )

        return found


In [ ]:
if RUN_PREPROCESSING:
    def extract_api_items(payload: Any) -> list[dict]:
        """
        다음과 같은 공공데이터 응답을 모두 처리합니다.

        response.body.items.item
        response.body.items
        body.items.item
        body.items
        items.item
        items
        """
        def normalize_item_value(value):
            if isinstance(value, list):
                return [
                    item for item in value
                    if isinstance(item, dict)
                ]

            if isinstance(value, dict):
                # items = {"item": [...]}
                for key in ("item", "row", "data", "list"):
                    nested = value.get(key)

                    if isinstance(nested, list):
                        return [
                            item for item in nested
                            if isinstance(item, dict)
                        ]

                    if isinstance(nested, dict):
                        return [nested]

                # dict 자체가 단일 성분 item일 가능성
                scalar_values = [
                    item for item in value.values()
                    if not isinstance(item, (dict, list))
                ]
                if scalar_values:
                    return [value]

            return []

        if isinstance(payload, dict):
            candidate_nodes = [payload]

            response_node = payload.get("response")
            if isinstance(response_node, dict):
                candidate_nodes.append(response_node)

            body_node = payload.get("body")
            if isinstance(body_node, dict):
                candidate_nodes.append(body_node)

            if isinstance(response_node, dict):
                nested_body = response_node.get("body")
                if isinstance(nested_body, dict):
                    candidate_nodes.append(nested_body)

            for node in candidate_nodes:
                for key in (
                    "items",
                    "item",
                    "data",
                    "rows",
                    "row",
                    "list",
                ):
                    if key in node:
                        items = normalize_item_value(
                            node[key]
                        )
                        if items:
                            return items

        # 마지막 수단: 모든 중첩 리스트 검색
        candidates = recursive_find_lists(payload)

        if candidates:
            candidates.sort(
                key=lambda pair: (
                    0 if re.search(
                        r"items?|data|rows?|list",
                        pair[0],
                        re.I,
                    ) else 1,
                    -len(pair[1]),
                )
            )
            return candidates[0][1]

        return []


In [ ]:
if RUN_PREPROCESSING:
    def recursive_find_scalar(
        obj: Any,
        wanted_keys: set[str],
    ) -> Any:
        if isinstance(obj, dict):
            for key, value in obj.items():
                if (
                    comparison_key(key) in wanted_keys
                    and not isinstance(value, (dict, list))
                ):
                    return value

            for value in obj.values():
                result = recursive_find_scalar(
                    value,
                    wanted_keys,
                )
                if result is not None:
                    return result

        elif isinstance(obj, list):
            for value in obj:
                result = recursive_find_scalar(
                    value,
                    wanted_keys,
                )
                if result is not None:
                    return result

        return None


    def extract_total_count(payload: Any) -> int | None:
        value = recursive_find_scalar(
            payload,
            {
                "totalcount",
                "totalcnt",
                "totcnt",
                "total",
            },
        )

        try:
            return int(str(value).replace(",", ""))
        except (TypeError, ValueError):
            return None


In [ ]:
if RUN_PREPROCESSING:
    def extract_result_summary(payload: Any) -> dict[str, Any]:
        return {
            "code": recursive_find_scalar(
                payload,
                {
                    "resultcode",
                    "errcode",
                    "code",
                    "returnreasoncode",
                },
            ),
            "message": recursive_find_scalar(
                payload,
                {
                    "resultmsg",
                    "errmsg",
                    "message",
                    "returnauthmsg",
                },
            ),
            "total_count": extract_total_count(payload),
        }


    def compact_payload_preview(
        payload: Any,
        limit: int = 800,
    ) -> str:
        try:
            preview = json.dumps(
                payload,
                ensure_ascii=False,
                default=str,
            )
        except Exception:
            preview = str(payload)

        preview = re.sub(r"\s+", " ", preview)
        return preview[:limit]


In [ ]:
if RUN_PREPROCESSING:
    def discover_mfds_endpoint(
        base_url: str,
        api_key: str,
        operation_path: str | None = None,
        operation_candidates: list[str] | None = None,
        timeout: int = 30,
    ) -> tuple[
        str,
        dict[str, str],
        tuple[str, str],
        Any,
    ]:
        """
        실제 item이 반환되는 조합만 성공으로 인정합니다.
        """
        decoded_key = unquote(api_key.strip())

        operations = (
            [operation_path]
            if operation_path
            else (
                operation_candidates
                or DEFAULT_MFDS_OPERATION_CANDIDATES
            )
        )

        attempts = []

        for operation in operations:
            endpoint = (
                base_url.rstrip("/")
                + "/"
                + operation.lstrip("/")
            )

            for page_key, row_key in (
                DEFAULT_PAGING_PARAM_CANDIDATES
            ):
                for type_params in (
                    DEFAULT_RESPONSE_TYPE_PARAM_CANDIDATES
                ):
                    params = {
                        "serviceKey": decoded_key,
                        page_key: 1,
                        row_key: 10,
                        **type_params,
                    }

                    try:
                        response = requests.get(
                            endpoint,
                            params=params,
                            timeout=timeout,
                        )
                        response.raise_for_status()

                        payload = parse_api_payload(response)
                        items = extract_api_items(payload)
                        summary = extract_result_summary(
                            payload
                        )

                        attempts.append({
                            "endpoint": endpoint,
                            "page_params": (
                                page_key,
                                row_key,
                            ),
                            "type_params": type_params,
                            "status": response.status_code,
                            "items": len(items),
                            **summary,
                            "preview": compact_payload_preview(
                                payload,
                                400,
                            ),
                        })

                        # 중요: 실제 데이터가 있을 때만 확정
                        if items:
                            print(
                                "MFDS API 엔드포인트 확인:",
                                endpoint,
                            )
                            print(
                                "페이지 파라미터:",
                                {
                                    page_key: 1,
                                    row_key: 10,
                                },
                            )
                            print(
                                "응답 형식 파라미터:",
                                type_params
                                or "기본 응답(XML 가능)",
                            )
                            print(
                                "테스트 데이터 건수:",
                                len(items),
                            )

                            return (
                                endpoint,
                                type_params,
                                (page_key, row_key),
                                payload,
                            )

                    except Exception as exc:
                        attempts.append({
                            "endpoint": endpoint,
                            "page_params": (
                                page_key,
                                row_key,
                            ),
                            "type_params": type_params,
                            "status": "ERROR",
                            "items": 0,
                            "code": "",
                            "message": (
                                f"{type(exc).__name__}: "
                                f"{exc}"
                            ),
                            "total_count": "",
                            "preview": "",
                        })

        attempts_df = pd.DataFrame(attempts)

        print(
            "MFDS 엔드포인트 자동 확인 결과 "
            "(마지막 30개 시도)"
        )
        display(attempts_df.tail(30))

        raise RuntimeError(
            "실제 성분 item을 반환하는 MFDS API "
            "조합을 찾지 못했습니다.\n\n"
            "위 표에서 result code/message와 preview를 "
            "확인하세요.\n"
            "Swagger에서 상세기능 경로가 확인되면 "
            "MFDS_OPERATION_PATH에 직접 입력하세요."
        )


In [ ]:
if RUN_PREPROCESSING:
    def download_all_mfds_ingredients(
        base_url: str,
        api_key: str,
        operation_path: str | None = None,
        page_size: int = 100,
        max_pages: int = 1000,
        request_sleep: float = 0.15,
        timeout: int = 60,
    ) -> tuple[pd.DataFrame, dict[str, Any]]:
        (
            endpoint,
            type_params,
            paging_keys,
            first_payload,
        ) = discover_mfds_endpoint(
            base_url=base_url,
            api_key=api_key,
            operation_path=operation_path,
            timeout=timeout,
        )

        decoded_key = unquote(api_key.strip())
        page_key, row_key = paging_keys

        all_items = []
        total_count = None
        first_raw_columns = []

        for page_no in range(1, max_pages + 1):
            params = {
                "serviceKey": decoded_key,
                page_key: page_no,
                row_key: page_size,
                **type_params,
            }

            response = requests.get(
                endpoint,
                params=params,
                timeout=timeout,
            )
            response.raise_for_status()

            payload = parse_api_payload(response)
            items = extract_api_items(payload)

            if total_count is None:
                total_count = extract_total_count(
                    payload
                )

            if not items:
                summary = extract_result_summary(
                    payload
                )

                print(
                    f"{page_no}페이지에서 item이 "
                    "0건입니다.",
                    summary,
                )
                print(
                    "응답 미리보기:",
                    compact_payload_preview(
                        payload,
                        1000,
                    ),
                )
                break

            if page_no == 1:
                first_raw_columns = sorted(
                    set().union(
                        *[
                            item.keys()
                            for item in items
                            if isinstance(item, dict)
                        ]
                    )
                )

            all_items.extend(items)

            print(
                f"MFDS {page_no}페이지: "
                f"{len(items):,}건 / "
                f"누적 {len(all_items):,}건"
            )

            if (
                total_count is not None
                and len(all_items) >= total_count
            ):
                break

            if (
                total_count is None
                and len(items) < page_size
            ):
                break

            time.sleep(request_sleep)

        if not all_items:
            raise RuntimeError(
                "MFDS API가 정상 응답은 했지만 "
                "실제 성분 item은 0건입니다.\n"
                "위에 출력된 응답 미리보기를 확인하세요."
            )

        api_df = pd.DataFrame(all_items)

        meta = {
            "endpoint": endpoint,
            "response_type_params": type_params,
            "paging_params": {
                "page": page_key,
                "rows": row_key,
            },
            "total_count_reported": total_count,
            "downloaded_rows": len(api_df),
            "downloaded_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),
            "raw_columns": (
                first_raw_columns
                or list(api_df.columns)
            ),
        }

        return api_df, meta



## 셀 6. API 응답 컬럼 탐지와 표준 성분사전 생성

API 응답에서 다음 정보를 자동 탐지합니다.

- 표준 한글명
- 영문명
- CAS No.
- 이명
- 기원 및 정의
- 성분코드

API 필드명이 예상과 다르면 현재 응답 컬럼 목록을 출력하고 중단합니다.  
그 경우 설정 셀의 `MFDS_FIELD_OVERRIDES`에 실제 필드명을 한 번만 입력하면 됩니다.

이명은 `/`나 쉼표로 무조건 분리하지 않습니다. 공식 성분명 자체에 `/`와 쉼표가 포함될 수 있기 때문입니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    # -----------------------------------------------------------------------------
    # MFDS field detection and master construction v3.2
    # -----------------------------------------------------------------------------

    FIELD_CANDIDATES = {
        "standard_name": [
            "표준명칭", "표준명", "한글명", "성분명", "원료명",
            "INGR_KOR_NAME", "INGR_KOR_NM", "INGR_NM", "KOR_NM",
            "STD_NM", "MATERIAL_NAME_KOR",
        ],
        "english_name": [
            "영문명", "표준영문명", "INCI명", "INCI Name",
            "INGR_ENG_NAME", "INGR_ENG_NM", "ENG_NM", "MATERIAL_NAME_ENG",
        ],
        "cas_no": ["CASNo", "CAS No", "CAS_NO", "CASNO", "CAS 번호"],
        "alias": [
            "이명", "동의어", "구명칭", "이전명칭", "SYNONYM", "ALIAS",
            "INGR_SYNONYM", "OLD_NM", "INGR_OLD_NM", "ETC_NM",
        ],
        "origin_definition": [
            "기원및정의", "기원 및 정의", "기원", "정의",
            "ORIGIN_MAJOR_KOR_NAME", "ORIGIN_DEF", "ORIGIN",
        ],
        "ingredient_code": [
            "성분코드", "원료코드", "INGR_CODE", "INGR_CD", "ITEM_SEQ", "SEQ",
        ],
    }


In [ ]:
if RUN_PREPROCESSING:
    def detect_column(
        columns: Iterable[str],
        candidate_names: list[str],
        override: str | None = None,
    ) -> str | None:
        columns = list(columns)

        if override:
            if override not in columns:
                raise KeyError(
                    f"지정한 API 컬럼이 없습니다: {override}\n"
                    f"실제 컬럼: {columns}"
                )
            return override

        key_to_original = {
            comparison_key(column): column
            for column in columns
        }

        for candidate in candidate_names:
            key = comparison_key(candidate)
            if key in key_to_original:
                return key_to_original[key]

        # 보수적인 부분일치. 너무 짧은 후보는 제외합니다.
        for candidate in candidate_names:
            key = comparison_key(candidate)
            if len(key) < 5:
                continue
            for column in columns:
                column_key = comparison_key(column)
                if key in column_key or column_key in key:
                    return column

        return None


    def split_alias_names(value: Any) -> list[str]:
        text = normalize_spaces(value)
        if not text:
            return []

        # 쉼표와 /는 공식 성분명 자체에 포함될 수 있으므로 분리자로 쓰지 않습니다.
        return [
            part.strip()
            for part in re.split(r"[;|\n]+", text)
            if part.strip()
        ]


In [ ]:
if RUN_PREPROCESSING:
    def normalize_mfds_master(
        api_df: pd.DataFrame,
        field_overrides: dict[str, str | None] | None = None,
    ) -> tuple[pd.DataFrame, dict[str, str | None]]:
        overrides = field_overrides or {}

        detected = {
            logical: detect_column(
                api_df.columns,
                candidates,
                overrides.get(logical),
            )
            for logical, candidates in FIELD_CANDIDATES.items()
        }

        standard_col = detected["standard_name"]
        if standard_col is None:
            raise KeyError(
                "MFDS API 응답에서 표준명 컬럼을 찾지 못했습니다.\n"
                f"API 컬럼: {list(api_df.columns)}\n"
                "MFDS_FIELD_OVERRIDES['standard_name']에 실제 컬럼명을 입력하세요."
            )

        def normalized_column(logical_name: str):
            column = detected[logical_name]
            if column is None:
                return pd.Series("", index=api_df.index, dtype="object")
            return api_df[column].map(normalize_spaces)

        master = pd.DataFrame({
            "standard_name": api_df[standard_col].map(normalize_spaces),
            "english_name": normalized_column("english_name"),
            "cas_no": normalized_column("cas_no"),
            "aliases": normalized_column("alias"),
            "origin_definition": normalized_column("origin_definition"),
            "ingredient_code": normalized_column("ingredient_code"),
        })

        master = master[master["standard_name"].ne("")].copy()
        master = master.drop_duplicates(
            subset=[
                "standard_name", "english_name", "cas_no",
                "aliases", "origin_definition", "ingredient_code",
            ]
        ).reset_index(drop=True)

        master["dictionary_source"] = "MFDS_OPEN_API"

        return master, detected


In [ ]:
if RUN_PREPROCESSING:
    def build_ingredient_maps(
        master_df: pd.DataFrame,
        manual_aliases: dict[str, str],
    ):
        canonical_map: dict[str, str] = {}
        metadata_map: dict[str, dict[str, str]] = {}
        alias_candidates: dict[str, set[str]] = defaultdict(set)

        for row in master_df.itertuples(index=False):
            canonical = normalize_spaces(row.standard_name)
            if not canonical:
                continue

            key = comparison_key(canonical)
            canonical_map[key] = canonical

            previous = metadata_map.get(key, {})
            metadata_map[key] = {
                "ingredient_code": (
                    normalize_spaces(getattr(row, "ingredient_code", ""))
                    or previous.get("ingredient_code", "")
                ),
                "canonical_english_name": (
                    normalize_spaces(getattr(row, "english_name", ""))
                    or previous.get("canonical_english_name", "")
                ),
                "cas_no": (
                    normalize_spaces(getattr(row, "cas_no", ""))
                    or previous.get("cas_no", "")
                ),
            }

            for alias in split_alias_names(getattr(row, "aliases", "")):
                alias_key = comparison_key(alias)
                if alias_key and alias_key != key:
                    alias_candidates[alias_key].add(canonical)

        # 하나의 이명이 여러 표준명에 연결되면 자동 매핑하지 않습니다.
        alias_map = {
            alias_key: next(iter(candidates))
            for alias_key, candidates in alias_candidates.items()
            if len(candidates) == 1
        }

        ambiguous_alias_map = {
            alias_key: sorted(candidates)
            for alias_key, candidates in alias_candidates.items()
            if len(candidates) > 1
        }

        manual_map = {
            comparison_key(raw): canonical
            for raw, canonical in manual_aliases.items()
        }

        return (
            canonical_map,
            alias_map,
            manual_map,
            metadata_map,
            ambiguous_alias_map,
        )



## 셀 7. API 인증키 로딩 및 캐시 준비

실행 방식:

- 캐시가 없거나 `REFRESH_MFDS_CACHE=True`: API 전체 다운로드
- 캐시가 있고 `REFRESH_MFDS_CACHE=False`: 기존 캐시 즉시 사용
- API 원본과 정규화된 성분사전을 `_cache` 폴더에 저장

인증키는 노트북 출력에 표시하지 않습니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    import getpass


    def get_mfds_api_key():
        if MFDS_API_KEY_DIRECT.strip():
            return MFDS_API_KEY_DIRECT.strip()

        try:
            from google.colab import userdata
            secret_key = userdata.get("MFDS_API_KEY")
            if secret_key:
                return secret_key
        except Exception:
            pass

        return getpass.getpass(
            "공공데이터포털 MFDS API 인증키 입력: "
        ).strip()


    mfds_master_df = pd.DataFrame()
    mfds_api_meta = {}


In [ ]:
if RUN_PREPROCESSING:
    if USE_MFDS_API:
        master_cache_valid = (
            MFDS_MASTER_CACHE_PATH.exists()
            and MFDS_MASTER_CACHE_PATH.stat().st_size > 20
        )
        raw_cache_valid = (
            MFDS_RAW_CACHE_PATH.exists()
            and MFDS_RAW_CACHE_PATH.stat().st_size > 20
        )

        if master_cache_valid and not REFRESH_MFDS_CACHE:
            print("기존 MFDS 표준 성분사전 캐시를 사용합니다.")
            mfds_master_df = pd.read_csv(
                MFDS_MASTER_CACHE_PATH,
                encoding="utf-8-sig",
                dtype=str,
            ).fillna("")

        elif raw_cache_valid and not REFRESH_MFDS_CACHE:
            print(
                "MFDS API 원본 캐시는 있으나 표준 캐시가 없어 "
                "원본 캐시에서 다시 생성합니다."
            )
            mfds_api_raw_df = pd.read_csv(
                MFDS_RAW_CACHE_PATH,
                encoding="utf-8-sig",
                dtype=str,
            ).fillna("")

            mfds_master_df, detected_fields = normalize_mfds_master(
                mfds_api_raw_df,
                field_overrides=MFDS_FIELD_OVERRIDES,
            )
            mfds_master_df.to_csv(
                MFDS_MASTER_CACHE_PATH,
                index=False,
                encoding="utf-8-sig",
            )
            mfds_api_meta["detected_fields"] = detected_fields

        else:
            print("MFDS API에서 전체 성분사전을 내려받습니다.")
            api_key = get_mfds_api_key()

            mfds_api_raw_df, mfds_api_meta = (
                download_all_mfds_ingredients(
                    base_url=MFDS_API_BASE_URL,
                    api_key=api_key,
                    operation_path=MFDS_OPERATION_PATH,
                    page_size=MFDS_PAGE_SIZE,
                )
            )

            mfds_api_raw_df.to_csv(
                MFDS_RAW_CACHE_PATH,
                index=False,
                encoding="utf-8-sig",
            )

            print("API 원본 컬럼:")
            print(list(mfds_api_raw_df.columns))

            mfds_master_df, detected_fields = normalize_mfds_master(
                mfds_api_raw_df,
                field_overrides=MFDS_FIELD_OVERRIDES,
            )
            mfds_api_meta["detected_fields"] = detected_fields

            mfds_master_df.to_csv(
                MFDS_MASTER_CACHE_PATH,
                index=False,
                encoding="utf-8-sig",
            )

            MFDS_META_CACHE_PATH.write_text(
                json.dumps(
                    mfds_api_meta,
                    ensure_ascii=False,
                    indent=2,
                ),
                encoding="utf-8",
            )
            print("MFDS 캐시 저장 완료")

        if MFDS_META_CACHE_PATH.exists() and not mfds_api_meta:
            try:
                mfds_api_meta = json.loads(
                    MFDS_META_CACHE_PATH.read_text(
                        encoding="utf-8"
                    )
                )
            except Exception:
                mfds_api_meta = {}

        required_master_columns = {
            "standard_name", "english_name", "cas_no",
            "aliases", "origin_definition", "ingredient_code",
        }
        missing_master_columns = (
            required_master_columns - set(mfds_master_df.columns)
        )
        if missing_master_columns:
            raise RuntimeError(
                "MFDS 표준 캐시 컬럼이 올바르지 않습니다: "
                f"{sorted(missing_master_columns)}"
            )

        print(f"MFDS 표준 성분 수: {len(mfds_master_df):,}")
        print(
            "MFDS 표준명 고유 수:",
            mfds_master_df["standard_name"].nunique(),
        )
        display(mfds_master_df.head(5))

        if mfds_api_meta:
            print("API 사용 정보")
            display(pd.DataFrame([mfds_api_meta]))
    else:
        print("MFDS API 사용이 꺼져 있습니다.")


MFDS API 원본 캐시는 있으나 표준 캐시가 없어 원본 캐시에서 다시 생성합니다.
MFDS 표준 성분 수: 21,833
MFDS 표준명 고유 수: 21832


,standard_name,english_name,cas_no,aliases,origin_definition,ingredient_code,dictionary_source
0,가공소금,,,,,,MFDS_OPEN_API
1,가지열매추출물,Solanum Melongena (Eggplant) Fruit Extract,84012-19-1,가지추출물,이 원료는 가지(Eggplant) Solanum melongena의 열매에서 추출한 것이다.,,MFDS_OPEN_API
2,구멍쇠미역추출물,Agarum Cribrosum Extract,,,이 원료는 조류의 일종인 구멍쇠미역 Agarum cribosum에서 추출한 것이다.,,MFDS_OPEN_API
3,루핀아미노산,Lupine Amino Acids,,,이 원료는 루핀 단백질의 완전 가수분해로 얻은 아미노산의 혼합물이다.,,MFDS_OPEN_API
4,류신,Leucine,"328-39-2(DL-) ,61-90-5(L-)",,이 원료는 다음의 구조를 갖는 아미노산이다.,,MFDS_OPEN_API


API 사용 정보


,detected_fields
0,"{'standard_name': 'INGR_KOR_NAME', 'english_name': 'INGR_ENG_NAME', 'cas_no': 'CAS_NO', 'alias': 'INGR_SYNONYM', 'origin_definition': 'ORIGIN_MAJOR_KOR_NAME', 'ingredient_code'..."



## 셀 8. 사용자 조정 가능한 성분 규칙

### 수동 Alias

MFDS API의 표준명·이명에서 자동으로 해결되지 않는, 사람이 동일 성분임을 확인한 경우만 추가합니다.

- 부분 문자열 치환이 아니라 **성분 전체 이름이 일치할 때만** 적용
- `D-판테놀 → 판테놀`처럼 화학형이 다를 가능성이 있는 항목은 임의 병합하지 않음
- MFDS에서 별도 표준명으로 확인되면 API 결과가 우선

### 공백형 보조 Lexicon

MFDS 사전에 포함되더라도 크롤링 표기 차이로 누락되는 사례를 보완합니다.  
여기 추가된 이름은 성분 분리를 돕는 용도이며, 표준명 자동 병합을 뜻하지 않습니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    MANUAL_ALIASES = {
        "1,2-헥산디올": "1,2-헥산다이올",
        "디메치콘": "다이메티콘",
        "디메치콘올": "다이메티콘올",
        "잔탐검": "잔탄검",
        "아르기닌": "알지닌",
        "베타글루칸": "베타-글루칸",
        "세틸알콜": "세틸알코올",
        "세테아릴알콜": "세테아릴알코올",
        "베헤닐알콜": "베헤닐알코올",
        "연필향나무오일": "연필향나무목부오일",
        "C13-14아이소파라핀": "C13-14아이소알케인",
        "폴리리신": "폴리라이신",
        "나이아산아마이드": "나이아신아마이드",
        "아시아티코사이두": "아시아티코사이드",
        "솔리탄올리베이트": "솔비탄올리베이트",
        "다이카프릴릴에테르": "다이카프릴릴에터",
        "리놀렌익애씨드": "리놀레닉애씨드",
        "리노레익애씨드": "리놀레익애씨드",
        "12-헥산다이올": "1,2-헥산다이올",
        "글루타킥애씨드": "글루타믹애씨드",
        "피토스광고신": "피토스핑고신",
        "로즈힙열매추출": "로즈힙열매추출물",
    }

    EXTRA_INGREDIENT_LEXICON = [
        "세틸에스터",
        "그린와틀꽃왁스",
        "세테아릴아이소노나노에이트",
        "아크릴아마이드/소듐아크릴로일다이메틸타우레이트코폴리머",
        "피이지-8스테아레이트",
        "미리스틸미리스테이트",
        "하이드록시팔미토일스핑가닌",
        "콜로이달오트밀",
        "C15-19알케인",
        "프로방스장미꽃수",
        "프로방스장미꽃추출물",
        "라우릴라우레이트",
        "적색 504호",
        "비스-피이지-18메틸에터다이메틸실레인",
        "아이소프로필아이소스테아레이트",
        "갈랑가갈랑갈잎추출물",
        "파이틱애씨드",
        "폴리엡실론-라이신",
        "C13-14아이소알케인",
    ]



## 셀 9. 상품명·함량·설명문 전처리 함수

주요 수정사항:

- `전성분 정제수` → `정제수`
- `스쿠알란(150,000ppm)` → 성분명과 함량 분리
- 마침표로 구분된 전성분을 정상 분리
- `손상된 피부`, `사용방법`, `*3X RETINOL` 이후 설명문 제거
- 잘못 닫힌 `[크림}` 표기를 `[크림]`으로 보정


In [ ]:
if RUN_PREPROCESSING:
    pass
    # -----------------------------------------------------------------------------
    # Product and ingredient text preprocessing v3.2
    # -----------------------------------------------------------------------------

    CONCENTRATION_PAREN_RE = re.compile(
        r"\(\s*([0-9][0-9,]*(?:\.[0-9]+)?)\s*(ppm|ppb|%)\s*\)",
        re.I,
    )
    CONCENTRATION_TRAILING_RE = re.compile(
        r"\s+([0-9][0-9,]*(?:\.[0-9]+)?)\s*(ppm|ppb|%)\s*$",
        re.I,
    )
    CONCENTRATION_ATTACHED_RE = re.compile(
        r"\s*\*?\s*([0-9][0-9,]*(?:\.[0-9]+)?)\s*(ppm|ppb|%)\s*(?:함유)?\s*$",
        re.I,
    )

    DESCRIPTION_STOP_PATTERNS = [
        r"\s+손상된\s*피부",
        r"\s+사용\s*방법",
        r"\s+사용법",
        r"\s+용법\s*[·ㆍ]?\s*용량",
        r"\s+효능\s*[·ㆍ]?\s*효과",
        r"\s+주의\s*사항",
        r"\s+본\s*제품은",
        r"\s*\*\s*3X\s+RETINOL",
        r"\s*\*\s*활성[-\s]",
        r"\s+\*+표시\s*[:：]",
        r"\s+\+표시\s*[:：]",
        r"\s*※\s*※?\s*자연유래",
        r"\s*※\s*자연유래",
    ]

    CHOICE_PRODUCT_TERMS = [
        "택1", "택 1", "골라담기", "옵션선택", "옵션 선택",
        "중 1종", "중1종", "2종 중", "두 가지 중",
    ]

    TARGET_FORM_TERMS = [
        "겔크림", "젤크림", "수딩크림", "크림", "밤", "연고",
    ]

    EXCLUDED_FORM_TERMS = [
        "증정품", "증정", "사은품", "세럼", "토너", "미스트",
        "에센스", "패드", "클렌저", "클렌징", "앰플", "마스크",
        "로션", "선크림", "폼", "샴푸", "패치", "아이크림",
        "아이앤링클", "크림마스크", "멀티밤",
    ]

    COMPONENT_TERMS = [
        "젤", "캡슐", "1제", "2제", "내상", "외상",
    ]

    PREFERRED_SECTION_TERMS = [
        "본품", "리뉴얼 적용", "메인",
    ]


In [ ]:
if RUN_PREPROCESSING:
    REJECT_SECTION_TERMS = [
        "리뉴얼 미적용", "구성품", "증정용",
    ]

    COMMON_INGREDIENT_ANCHORS = [
        "정제수", "온천수", "글리세린", "부틸렌글라이콜",
        "프로판다이올", "다이메티콘", "스쿠알란",
        "나이아신아마이드", "카프릴릭/카프릭트라이글리세라이드",
        "하이드로제네이티드폴리아이소부텐", "편백수", "병풀잎수",
        "하이드롤라이즈드하이알루로닉애씨드",
    ]

    # 크롤링 과정에서 실제 확인된 명확한 문자 오염만 보정합니다.
    KNOWN_TEXT_REPAIRS = {
        "암X5:AD5모늄": "암모늄",
        "암x5:ad5모늄": "암모늄",
    }

    SECONDARY_INLINE_FORM_RE = re.compile(
        r"\s+(?:[A-Za-z0-9가-힣\-]+\s*){0,5}"
        r"(?:카밍패드|패드|세럼|앰플|토너|미스트|에센스|패치|멀티밤)"
        r"\s*[:：]?\s*"
        r"(?=(?:정제수|글리세린|부틸렌글라이콜|프로판다이올|나이아신아마이드))",
        re.I,
    )


    def clean_product_name(product_name: Any) -> str:
        text = normalize_spaces(product_name)
        text = re.sub(r"\[[^\]]*\]", " ", text)
        text = re.sub(
            r"\([^)]*(?:증정|기획|추가|사은품)[^)]*\)",
            " ",
            text,
            flags=re.I,
        )
        text = re.sub(
            r"\b\d+(?:\.\d+)?\s*(?:ml|g|ea|매)\b",
            " ",
            text,
            flags=re.I,
        )
        text = re.sub(r"\b\d+\s*(?:입|개입|개|팩)\b", " ", text)
        text = re.sub(r"\b\d+\s*\+\s*\d+\b", " ", text)

        for term in [
            "기획세트", "더블기획", "단독기획", "온라인단독", "한정기획",
            "기획", "단품", "더블팩", "더블", "1+1", "증정", "본품",
        ]:
            text = text.replace(term, " ")

        text = re.sub(r"[+&,]", " ", text)
        text = re.sub(r"\(\s*\)", " ", text)

        return re.sub(r"\s+", " ", text).strip()


In [ ]:
if RUN_PREPROCESSING:
    def normalize_raw_ingredient_text(value: Any) -> str:
        text = normalize_spaces(value)

        # 전체 문자열을 감싼 따옴표 제거
        text = text.strip('"\'“”‘’')

        for wrong, correct in KNOWN_TEXT_REPAIRS.items():
            text = text.replace(wrong, correct)

        # 잘못 닫힌 섹션 표기: [크림} -> [크림]
        text = re.sub(r"\[([^\]\{\}]{1,80})\}", r"[\1]", text)

        # 마침표가 성분 구분자로 쓰인 경우. 소수점은 건드리지 않습니다.
        text = re.sub(
            r"(?<=[가-힣A-Za-z\)])\.\s*(?=[가-힣A-Za-zC])",
            ", ",
            text,
        )
        # 폴리쿼터늄-51. 처럼 숫자로 끝나는 성분의 마침표 구분자
        # 1. 제품명 / 2. 제품명 형식의 번호 섹션은 건드리지 않습니다.
        text = re.sub(
            r"([가-힣A-Za-z][가-힣A-Za-z0-9/\-]*-\d+)\.\s*(?=[가-힣A-Za-zC])",
            r"\1, ",
            text,
        )

        # 설명문·마케팅 문구 이후는 성분표가 아닙니다.
        stop_positions = []
        for pattern in DESCRIPTION_STOP_PATTERNS:
            match = re.search(pattern, text, flags=re.I)
            if match:
                stop_positions.append(match.start())

        if stop_positions:
            text = text[:min(stop_positions)].strip()

        return text.rstrip(" (,;:-.\"'“”‘’")


In [ ]:
if RUN_PREPROCESSING:
    def is_choice_product(
        product_name: str,
        raw_text: str = "",
    ) -> tuple[bool, str]:
        normalized = comparison_key(product_name)

        for term in CHOICE_PRODUCT_TERMS:
            if comparison_key(term) in normalized:
                return True, f"선택상품 키워드: {term}"

        # 광고용 대괄호 안의 슬래시는 옵션 구분에서 제외합니다.
        option_name = re.sub(r"\[[^\]]*\]", " ", product_name)

        if "/" in option_name:
            slash_parts = [
                normalize_spaces(part)
                for part in option_name.split("/")
                if normalize_spaces(part)
            ]
            target_parts = [
                part
                for part in slash_parts
                if any(term in part for term in TARGET_FORM_TERMS)
            ]
            if len(target_parts) >= 2:
                return True, "슬래시로 구분된 복수 크림류 상품"

        for content in re.findall(r"\(([^)]{1,120})\)", option_name):
            if "/" in content and sum(
                content.count(term)
                for term in TARGET_FORM_TERMS
            ) >= 2:
                return True, "괄호 안 복수 크림류 옵션"

        return False, ""


    def normalize_ingredient_spacing(text: str) -> str:
        compact = re.sub(r"\s+", "", text)
        compact = re.sub(
            r"^(청색|적색|황색|녹색)(\d+)호$",
            r"\1 \2호",
            compact,
        )
        return compact


In [ ]:
if RUN_PREPROCESSING:
    def strip_leading_product_label(text: str) -> str:
        """PDRN 수분크림 정제수,... 형태에서 제품명 접두부만 제거합니다."""
        positions = [
            text.find(anchor)
            for anchor in COMMON_INGREDIENT_ANCHORS
            if text.find(anchor) >= 0
        ]

        if not positions:
            return text

        first_position = min(positions)
        if not (0 < first_position <= 120):
            return text

        prefix = text[:first_position]
        prefix_key = comparison_key(prefix)

        product_label_hint = (
            any(term in prefix for term in TARGET_FORM_TERMS)
            or any(term in prefix for term in EXCLUDED_FORM_TERMS)
            or "pdrn" in prefix_key
            or re.search(r"\d+(?:\.\d+)?\s*(?:ml|g)\b", prefix, re.I)
        )

        if product_label_hint:
            return text[first_position:].strip()

        return text


    def strip_inline_secondary_product(text: str) -> tuple[str, str]:
        """본품 성분 뒤에 구분자 없이 붙은 패드/앰플/세럼 등을 절단합니다."""
        match = SECONDARY_INLINE_FORM_RE.search(text)

        if match and match.start() > 20:
            removed = text[match.start():].strip()
            return text[:match.start()].rstrip(" ,;:-"), removed

        return text, ""


    def prepare_component_body(
        body: Any,
        section_label: str = "",
        product_name: str = "",
    ) -> tuple[str, str]:
        text = normalize_raw_ingredient_text(body)
        text, removed_secondary = strip_inline_secondary_product(text)
        text = strip_leading_product_label(text)
        text = re.sub(r"^(?:전성분|주성분|성분)\s*[:：]?\s*", "", text)
        return text.strip(), removed_secondary


In [ ]:
if RUN_PREPROCESSING:
    def clean_ingredient_token(token: Any) -> str:
        text = normalize_spaces(token)
        text = text.strip('"\'“”‘’')

        for wrong, correct in KNOWN_TEXT_REPAIRS.items():
            text = text.replace(wrong, correct)

        text = re.sub(r"^[•●■◆▶○\-–—]+\s*", "", text)
        text = re.sub(r"^\d+[.)]\s*", "", text)
        text = re.sub(r"^(?:전성분|주성분|성분)\s*[:：]?\s*", "", text)

        # 명백한 제품명 접두부가 첫 성분에 붙은 경우
        text = strip_leading_product_label(text)

        # 별표 이후의 각주/브랜드 복합체 설명 제거
        text = re.sub(r"^\*+\s*", "", text)
        text = re.sub(r"(?<=\S)\s*\*+.*$", "", text)
        text = re.sub(r"\s*\++\s*$", "", text)
        text = text.rstrip(" .,:;\"'“”‘’")

        text = re.sub(r"\s+([,()/])", r"\1", text)
        text = re.sub(r"([(/])\s+", r"\1", text)

        return normalize_ingredient_spacing(text.strip(" ,;"))


In [ ]:
if RUN_PREPROCESSING:
    def extract_concentration(token: Any) -> dict[str, Any]:
        raw_token = normalize_spaces(token)
        raw_token = raw_token.strip('"\'“”‘’')

        matches = list(CONCENTRATION_PAREN_RE.finditer(raw_token))
        concentration_raw = ""
        concentration_value = np.nan
        concentration_unit = ""
        concentration_ppm = np.nan
        ingredient_part = raw_token
        multiple_flag = 0

        if matches:
            match = matches[-1]
            multiple_flag = int(len(matches) > 1)
            concentration_raw = match.group(0)
            concentration_value = float(match.group(1).replace(",", ""))
            concentration_unit = match.group(2).lower()
            ingredient_part = (
                raw_token[:match.start()]
                + " "
                + raw_token[match.end():]
            ).strip()
        else:
            match = CONCENTRATION_TRAILING_RE.search(raw_token)
            if match is None:
                match = CONCENTRATION_ATTACHED_RE.search(raw_token)

            if match:
                concentration_raw = match.group(0).strip()
                concentration_value = float(match.group(1).replace(",", ""))
                concentration_unit = match.group(2).lower()
                ingredient_part = raw_token[:match.start()].strip()

        if concentration_unit == "%":
            concentration_ppm = concentration_value * 10_000
        elif concentration_unit == "ppb":
            concentration_ppm = concentration_value / 1_000
        elif concentration_unit == "ppm":
            concentration_ppm = concentration_value

        return {
            "ingredient_name_clean": clean_ingredient_token(ingredient_part),
            "concentration_raw": concentration_raw,
            "concentration_value": concentration_value,
            "concentration_unit_original": concentration_unit,
            "concentration_ppm": concentration_ppm,
            "has_declared_concentration": int(bool(concentration_unit)),
            "multiple_concentration_flag": multiple_flag,
        }



## 셀 10. 본품·증정품·다중 옵션 섹션 파서

### 유지하는 유형

- 일반 크림 본품
- `리뉴얼 적용` 처방
- 젤+캡슐로 구성된 캡슐 크림
- 1제+2제로 구성된 하나의 크림
- 세트 상품에 토너·에멀전·크림이 각각 표시된 경우 크림 섹션만

### 제외하는 유형

- 세럼·토너·미스트·패치·앰플 등 증정품
- 크림 토너, 아이크림, 크림 마스크 등 대상 외 제형
- `리뉴얼 미적용`
- 2종 택1·서로 다른 크림 중 선택하는 상품 전체

다중 선택상품은 상품 리뷰가 서로 섞였을 가능성이 있어 X와 Y의 연결이 불명확하므로 Long 데이터에서 제외하고 로그에 남깁니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    # -----------------------------------------------------------------------------
    # Section parsing v3.2
    # -----------------------------------------------------------------------------

    @dataclass
    class Section:
        label: str
        body: str
        marker_type: str
        order: int


    def _first_anchor_position(text: str) -> int | None:
        positions = [
            text.find(anchor)
            for anchor in COMMON_INGREDIENT_ANCHORS
            if text.find(anchor) >= 0
        ]
        return min(positions) if positions else None


    def split_label_and_body(segment: str) -> tuple[str, str]:
        segment = normalize_spaces(segment).strip(" :-")
        anchor_pos = _first_anchor_position(segment)

        if anchor_pos is not None and anchor_pos > 0:
            return (
                segment[:anchor_pos].strip(" :-"),
                segment[anchor_pos:].strip(),
            )

        return "", segment


In [ ]:
if RUN_PREPROCESSING:
    def extract_explicit_sections(text: str) -> list[Section]:
        """
        지원 예:
        [크림] ... [세럼] ...
        (크림) ... (세럼) ...
        ● 크림 : ... ● 토너 : ...
        본품: ... 증정품: ...
        젤크림 : ... 카밍패드 : ...
        """
        form_words = [
            "겔크림", "젤크림", "수딩크림", "크림", "밤", "연고",
            "카밍패드", "패드", "세럼", "앰플", "토너", "미스트",
            "에센스", "패치", "멀티밤", "젤", "캡슐",
            "본품", "증정품", "사은품", "구성품",
        ]
        form_alt = "|".join(
            sorted(map(re.escape, form_words), key=len, reverse=True)
        )

        marker_pattern = re.compile(
            r"\[([^\]]{1,120})\]"
            r"|[●○■◆▶]\s*([^:：]{1,120})\s*[:：]"
            rf"|\(([^)]*?(?:{form_alt})[^)]{{0,80}})\)"
            r"|(?<![가-힣A-Za-z0-9])((?:본품|증정품|사은품|구성품))\s*[:：]"
            rf"|(?<![가-힣A-Za-z0-9])((?:[A-Za-z0-9가-힣\-]+\s*){{0,4}}(?:{form_alt}))\s*[:：]",
            flags=re.I,
        )

        matches = list(marker_pattern.finditer(text))
        if not matches:
            return []

        sections = []
        for index, match in enumerate(matches):
            label = next(
                (group for group in match.groups() if group),
                "",
            )
            start = match.end()
            end = (
                matches[index + 1].start()
                if index + 1 < len(matches)
                else len(text)
            )
            body = text[start:end].strip(" :-")

            if body:
                sections.append(
                    Section(
                        normalize_spaces(label),
                        body,
                        "EXPLICIT",
                        index + 1,
                    )
                )

        return sections


In [ ]:
if RUN_PREPROCESSING:
    def extract_numbered_sections(text: str) -> list[Section]:
        matches = list(
            re.finditer(r"(?<!\d)(?:^|\s)(\d+)[.)]\s+", text)
        )

        if len(matches) < 2:
            return []

        sections = []
        for index, match in enumerate(matches):
            start = match.end()
            end = (
                matches[index + 1].start()
                if index + 1 < len(matches)
                else len(text)
            )
            segment = text[start:end].strip()
            label, body = split_label_and_body(segment)
            sections.append(
                Section(
                    label or f"SECTION_{match.group(1)}",
                    body,
                    "NUMBERED",
                    index + 1,
                )
            )

        return sections


    def extract_dash_sections(text: str) -> list[Section]:
        matches = list(re.finditer(r"(?:^|\s)-\s+", text))

        if len(matches) < 2:
            return []

        sections = []
        for index, match in enumerate(matches):
            start = match.end()
            end = (
                matches[index + 1].start()
                if index + 1 < len(matches)
                else len(text)
            )
            segment = text[start:end].strip()
            label, body = split_label_and_body(segment)
            sections.append(
                Section(
                    label or f"DASH_{index + 1}",
                    body,
                    "DASH",
                    index + 1,
                )
            )

        return sections


In [ ]:
if RUN_PREPROCESSING:
    def extract_nested_component_sections(body: str) -> list[Section]:
        # 1) 1제(내상) ... 2) 2제(외상) ...
        matches = list(
            re.finditer(
                r"(?:^|\s)([12])[.)]\s*((?:1제|2제)[^\s,]{0,30})\s*",
                body,
            )
        )

        if len(matches) < 2:
            return []

        sections = []
        for index, match in enumerate(matches):
            start = match.end()
            end = (
                matches[index + 1].start()
                if index + 1 < len(matches)
                else len(body)
            )
            label = normalize_spaces(
                match.group(2) or f"COMPONENT_{match.group(1)}"
            )
            sections.append(
                Section(
                    label,
                    body[start:end].strip(),
                    "COMPONENT",
                    index + 1,
                )
            )

        return sections


In [ ]:
if RUN_PREPROCESSING:
    def classify_section(label: str, product_name: str) -> str:
        name = normalize_spaces(label).lower()

        if (
            "본품/증정 동일" in name
            or "본품/증정동일" in name
        ) and any(term in name for term in TARGET_FORM_TERMS):
            return "preferred"

        if any(term in name for term in REJECT_SECTION_TERMS):
            return "exclude"

        # 제외 제형 우선. 크림 토너, 아이크림, 크림마스크를 잡습니다.
        if any(term in name for term in EXCLUDED_FORM_TERMS):
            return "exclude"

        if any(term in name for term in PREFERRED_SECTION_TERMS):
            return "preferred"

        # 젤크림/겔크림은 완제품 target이고, 단독 젤/캡슐만 component입니다.
        if any(term in name for term in TARGET_FORM_TERMS):
            return "target"

        if any(term in name for term in COMPONENT_TERMS):
            return "component"

        return "unknown"


    def _label_tokens(text: str) -> set[str]:
        normalized = normalize_spaces(text)
        normalized = re.sub(r"\[[^\]]*\]", " ", normalized)
        normalized = re.sub(r"\([^)]*\)", " ", normalized)
        tokens = re.findall(r"[A-Za-z0-9가-힣]+", normalized.lower())

        stop = {
            "크림", "젤크림", "겔크림", "수딩크림", "밤", "연고",
            "본품", "기획", "더블", "증정", "ml", "g",
        }
        return {token for token in tokens if token not in stop and len(token) >= 2}


In [ ]:
if RUN_PREPROCESSING:
    def target_section_relevance(
        section: Section,
        product_name: str,
    ) -> float:
        label = normalize_spaces(section.label)
        product = normalize_spaces(product_name)
        label_key = comparison_key(label)
        product_key = comparison_key(product)

        score = 0.0

        if label_key and label_key in product_key:
            score += 2.0

        label_tokens = _label_tokens(label)
        product_tokens = _label_tokens(product)

        if label_tokens:
            score += (
                len(label_tokens & product_tokens)
                / len(label_tokens)
            )

        score += 0.5 * SequenceMatcher(
            None,
            label_key,
            product_key,
        ).ratio()

        if any(term in label for term in PREFERRED_SECTION_TERMS):
            score += 1.0

        return score


In [ ]:
if RUN_PREPROCESSING:
    def choose_target_sections(
        product_name: str,
        raw_text: str,
    ) -> tuple[list[Section], dict[str, Any]]:
        text = normalize_raw_ingredient_text(raw_text)

        if comparison_key(text) in {
            "", "없음", "수집실패", "nan", "none",
        }:
            return [], {
                "status": "EMPTY_OR_CRAWL_FAILED",
                "reason": text or "EMPTY",
                "all_sections": [],
            }

        choice, choice_reason = is_choice_product(product_name, text)
        if choice:
            return [], {
                "status": "SKIPPED_MULTI_OPTION",
                "reason": choice_reason,
                "all_sections": [],
            }

        sections = extract_explicit_sections(text)
        if not sections:
            sections = extract_numbered_sections(text)
        if not sections:
            sections = extract_dash_sections(text)
        if not sections:
            sections = [
                Section("PRIMARY", text, "FALLBACK", 1)
            ]

        classified = [
            (section, classify_section(section.label, product_name))
            for section in sections
        ]

        preferred = [
            section for section, cls in classified
            if cls == "preferred"
        ]
        targets = [
            section for section, cls in classified
            if cls == "target"
        ]
        components = [
            section for section, cls in classified
            if cls == "component"
        ]

        if preferred:
            selected = [preferred[0]]

        elif targets:
            if len(targets) == 1:
                selected = targets
            else:
                exact_main = [
                    section
                    for section in targets
                    if comparison_key(section.label) in {
                        "크림", "메인크림", "본품크림",
                    }
                ]

                if len(exact_main) == 1:
                    selected = exact_main
                else:
                    scored = sorted(
                        [
                            (
                                target_section_relevance(section, product_name),
                                section,
                            )
                            for section in targets
                        ],
                        key=lambda pair: pair[0],
                        reverse=True,
                    )

                    best_score, best_section = scored[0]
                    second_score = scored[1][0]

                    # 상품명과 특정 섹션의 일치도가 확실할 때만 메인 섹션 선택
                    if best_score >= 1.1 and (
                        best_score - second_score >= 0.35
                    ):
                        selected = [best_section]
                    else:
                        return [], {
                            "status": "SKIPPED_MULTIPLE_TARGET_PRODUCTS",
                            "reason": (
                                "서로 다른 크림류 섹션이 2개 이상이며 "
                                "메인 제품을 확정하지 못함"
                            ),
                            "all_sections": [
                                (
                                    section.label,
                                    cls,
                                    round(
                                        target_section_relevance(
                                            section,
                                            product_name,
                                        ),
                                        3,
                                    ),
                                )
                                for section, cls in classified
                            ],
                        }

        elif components and any(
            term in product_name
            for term in ["캡슐", "듀얼", "2제", "플러스"]
        ):
            selected = components

        elif len(sections) == 1:
            selected = sections

        else:
            return [], {
                "status": "NO_TARGET_SECTION",
                "reason": "대상 크림 섹션을 판별하지 못함",
                "all_sections": [
                    (section.label, cls)
                    for section, cls in classified
                ],
            }

        expanded = []
        removed_secondary_sections = []

        for section in selected:
            body = section.body

            if any(
                term in section.label
                for term in PREFERRED_SECTION_TERMS
            ):
                next_number = re.search(r"\s+2[.)]\s+", body)
                if next_number:
                    body = body[:next_number.start()].strip()

            body, removed_secondary = prepare_component_body(
                body,
                section_label=section.label,
                product_name=product_name,
            )

            if removed_secondary:
                removed_secondary_sections.append(
                    removed_secondary[:300]
                )

            prepared_section = Section(
                section.label,
                body,
                section.marker_type,
                section.order,
            )

            nested = extract_nested_component_sections(
                prepared_section.body
            )

            if nested:
                expanded.extend(nested)
            else:
                expanded.append(prepared_section)

        return expanded, {
            "status": "OK",
            "reason": "",
            "all_sections": [
                (section.label, cls)
                for section, cls in classified
            ],
            "removed_secondary_sections": removed_secondary_sections,
        }



## 셀 11. 안전한 성분 Tokenizer

- 화학 위치번호 쉼표 보존: `1,2-헥산다이올`
- 천 단위 쉼표 보존: `150,000ppm`
- `/` 포함 공식 성분명 보존
- 문자열에 쉼표가 있어도 실제 성분이 2개 이상 분리될 때만 쉼표형으로 판단
- 공백형 전성분은 MFDS 표준명·이명 사전으로 최장일치 분리
- 공백형 성분 뒤의 `(0.1%)`도 같은 성분 토큰으로 처리


In [ ]:
if RUN_PREPROCESSING:
    pass
    # -----------------------------------------------------------------------------
    # Tokenizers
    # -----------------------------------------------------------------------------

    def safe_split_commas(text: str) -> list[str]:
        text = normalize_spaces(text)
        tokens = []
        buffer = []
        depth = 0
        index = 0
        while index < len(text):
            char = text[index]
            if char == "(":
                depth += 1
                buffer.append(char)
                index += 1
                continue
            if char == ")":
                depth = max(0, depth - 1)
                buffer.append(char)
                index += 1
                continue
            if char == "," and depth == 0:
                previous = text[index - 1] if index > 0 else ""
                following = text[index + 1:]
                current_fragment = "".join(buffer).strip()
                current_tail = re.split(r"\s+", current_fragment)[-1]
                is_chemical_locant = (
                    current_tail.isdigit()
                    and re.match(r"\s*\d+(?:,\d+)*-", following) is not None
                )
                is_numeric_thousands = previous.isdigit() and re.match(
                    r"\d{3}(?:,\d{3})*(?:\.\d+)?\s*(?:ppm|ppb|%)", following, flags=re.I
                ) is not None
                if is_chemical_locant or is_numeric_thousands:
                    buffer.append(char)
                    index += 1
                    continue
                token = "".join(buffer).strip()
                if token:
                    tokens.append(token)
                buffer = []
                index += 1
                continue
            buffer.append(char)
            index += 1
        final = "".join(buffer).strip()
        if final:
            tokens.append(final)
        return tokens


    def choose_tokenizer_method(text: str) -> str:
        if "@" in text:
            return "AT_DELIMITER"
        comma_tokens = safe_split_commas(text)
        if len(comma_tokens) >= 2:
            return "SAFE_COMMA"
        return "WHITESPACE_LEXICON"


In [ ]:
if RUN_PREPROCESSING:
    def tokenize_delimited(text: str, method: str) -> list[str]:
        if method == "AT_DELIMITER":
            return [token.strip() for token in text.split("@") if token.strip()]
        if method == "SAFE_COMMA":
            return safe_split_commas(text)
        raise ValueError(method)


    def build_ingredient_trie(names: Iterable[str]) -> dict:
        root = {}
        for display_name in names:
            display_name = normalize_spaces(display_name)
            compact = comparison_key(display_name)
            if not compact:
                continue
            node = root
            for char in compact:
                node = node.setdefault(char, {})
            node.setdefault("_end", []).append(display_name)
        return root


In [ ]:
if RUN_PREPROCESSING:
    def segment_with_ingredient_trie(text: str, trie: dict) -> tuple[list[str], list[str], float]:
        original = normalize_spaces(text)
        compact = comparison_key(original)
        length = len(compact)
        dp = [None] * (length + 1)
        dp[length] = (0, 0, 0, [])
        for start in range(length - 1, -1, -1):
            best = None
            node = trie
            end = start
            while end < length and compact[end] in node:
                node = node[compact[end]]
                end += 1
                if "_end" in node:
                    display_name = max(node["_end"], key=lambda value: len(comparison_key(value)))
                    # 공백형 전성분에서 '성분명(0.1%)' 함량 괄호까지 같은 토큰으로 소비
                    extended_end = end
                    suffix_match = re.match(r"\([0-9][0-9,]*(?:\.[0-9]+)?(?:ppm|ppb|%)\)", compact[end:], flags=re.I)
                    concentration_suffix = ""
                    if suffix_match:
                        concentration_suffix = suffix_match.group(0)
                        extended_end = end + len(concentration_suffix)
                    if dp[extended_end] is not None:
                        matched_len = extended_end - start
                        next_state = dp[extended_end]
                        display_token = display_name + concentration_suffix
                        candidate = (
                            next_state[0] + matched_len * 12 - 2,
                            next_state[1] + matched_len,
                            next_state[2] - 1,
                            [("MATCH", display_token, compact[start:extended_end])] + next_state[3],
                        )
                        if best is None or candidate[:3] > best[:3]:
                            best = candidate
            if dp[start + 1] is not None:
                next_state = dp[start + 1]
                candidate = (
                    next_state[0] - 18,
                    next_state[1],
                    next_state[2],
                    [("UNKNOWN", compact[start], compact[start])] + next_state[3],
                )
                if best is None or candidate[:3] > best[:3]:
                    best = candidate
            dp[start] = best
        path = dp[0][3] if dp[0] else []
        aggregated = []
        for kind, display, raw_piece in path:
            if kind == "UNKNOWN" and aggregated and aggregated[-1][0] == "UNKNOWN":
                prev = aggregated[-1]
                aggregated[-1] = ("UNKNOWN", prev[1] + display, prev[2] + raw_piece)
            else:
                aggregated.append((kind, display, raw_piece))
        tokens = [display for kind, display, _ in aggregated if kind == "MATCH"]
        unknown = [display for kind, display, _ in aggregated if kind == "UNKNOWN"]
        matched_chars = sum(len(comparison_key(display)) for kind, display, _ in aggregated if kind == "MATCH")
        coverage = matched_chars / max(1, len(compact))
        return tokens, unknown, coverage


In [ ]:
if RUN_PREPROCESSING:
    def expand_mixed_delimited_tokens(
        tokens: list[str],
        trie: dict,
        minimum_coverage: float = 0.995,
    ) -> tuple[list[str], int]:
        """쉼표 토큰 하나에 공백으로 두 성분이 붙은 경우 사전으로 재분리합니다."""
        expanded = []
        expanded_count = 0

        for token in tokens:
            if not re.search(r"\s", normalize_spaces(token)):
                expanded.append(token)
                continue

            segmented, unknown, coverage = segment_with_ingredient_trie(
                token,
                trie,
            )

            if (
                coverage >= minimum_coverage
                and not unknown
                and len(segmented) >= 2
            ):
                expanded.extend(segmented)
                expanded_count += 1
            else:
                expanded.append(token)

        return expanded, expanded_count



## 셀 12. 표준명 매핑·Long 변환·Formula Hash 파이프라인

매칭 우선순위:

1. MFDS 표준명 완전 일치
2. MFDS 이명 완전 일치
3. 승인된 수동 Alias
4. 불일치 시 원문 정규화 이름 보존 + 애매 로그

동일 성분표와 유사 상품명을 가진 다른 상품번호는 같은 `product_group_id`로 묶습니다.

원본 성분 중복은 그대로 보존합니다. 이후 원핫 생성 시 `sum`이 아닌 `max`를 사용해야 합니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    # -----------------------------------------------------------------------------
    # Canonical mapping and pipeline v3.2
    # -----------------------------------------------------------------------------


    def mapping_candidates(clean_name: str) -> list[tuple[str, str]]:
        """
        성분명 전체를 먼저 확인하고, 실패할 때만 안전한 괄호 제거 후보를 시도합니다.

        예:
        티타늄디옥사이드(CI77891) -> 티타늄디옥사이드
        온천수(아벤느온천수) -> 온천수

        하이드로제네이티드폴리(C6-14올레핀)는 전체명이 공식명으로 먼저 매칭되므로
        괄호가 임의로 삭제되지 않습니다.
        """
        candidates = [(clean_name, "DIRECT")]

        current = clean_name
        for _ in range(2):
            match = re.match(r"^(.*)\(([^()]*)\)$", current)
            if not match:
                break

            outer = match.group(1).strip()
            inner = match.group(2).strip()

            # 농도 괄호는 이미 extract_concentration에서 제거됩니다.
            if re.fullmatch(
                r"[0-9][0-9,]*(?:\.[0-9]+)?\s*(?:ppm|ppb|%)",
                inner,
                flags=re.I,
            ):
                break

            if outer:
                candidates.append((outer, "PAREN_FALLBACK"))
                current = outer
            else:
                break

        deduplicated = []
        seen = set()
        for candidate, method in candidates:
            key = comparison_key(candidate)
            if key and key not in seen:
                deduplicated.append((candidate, method))
                seen.add(key)

        return deduplicated


In [ ]:
if RUN_PREPROCESSING:
    def map_to_canonical(
        clean_name: str,
        canonical_map: dict[str, str],
        alias_map: dict[str, str],
        manual_map: dict[str, str],
        metadata_map: dict[str, dict[str, str]],
        ambiguous_alias_map: dict[str, list[str]],
        master_loaded: bool,
    ) -> dict[str, str]:
        ambiguous_candidates = []

        for candidate_name, candidate_method in mapping_candidates(clean_name):
            key = comparison_key(candidate_name)

            if key in canonical_map:
                canonical = canonical_map[key]
                meta = metadata_map.get(
                    comparison_key(canonical),
                    {},
                )
                return {
                    "canonical_name": canonical,
                    "ingredient_code": meta.get("ingredient_code", ""),
                    "canonical_english_name": meta.get(
                        "canonical_english_name", ""
                    ),
                    "cas_no": meta.get("cas_no", ""),
                    "mapping_status": (
                        "EXACT_MFDS"
                        if candidate_method == "DIRECT"
                        else "MFDS_PAREN_FALLBACK"
                    ),
                    "mapping_source": "MFDS",
                }

            if key in alias_map:
                canonical = alias_map[key]
                meta = metadata_map.get(
                    comparison_key(canonical),
                    {},
                )
                return {
                    "canonical_name": canonical,
                    "ingredient_code": meta.get("ingredient_code", ""),
                    "canonical_english_name": meta.get(
                        "canonical_english_name", ""
                    ),
                    "cas_no": meta.get("cas_no", ""),
                    "mapping_status": "MFDS_ALIAS",
                    "mapping_source": "MFDS",
                }

            if key in manual_map:
                canonical = manual_map[key]
                meta = metadata_map.get(
                    comparison_key(canonical),
                    {},
                )
                return {
                    "canonical_name": canonical,
                    "ingredient_code": meta.get("ingredient_code", ""),
                    "canonical_english_name": meta.get(
                        "canonical_english_name", ""
                    ),
                    "cas_no": meta.get("cas_no", ""),
                    "mapping_status": "MANUAL_ALIAS",
                    "mapping_source": "MANUAL",
                }

            if key in ambiguous_alias_map:
                ambiguous_candidates.extend(
                    ambiguous_alias_map[key]
                )

        return {
            "canonical_name": clean_name,
            "ingredient_code": "",
            "canonical_english_name": "",
            "cas_no": "",
            "mapping_status": (
                "AMBIGUOUS_MFDS_ALIAS"
                if ambiguous_candidates
                else (
                    "UNMATCHED_MFDS"
                    if master_loaded
                    else "NORMALIZED_ONLY"
                )
            ),
            "mapping_source": "NONE",
            "ambiguous_candidates": " | ".join(
                sorted(set(ambiguous_candidates))
            ),
        }


In [ ]:
if RUN_PREPROCESSING:
    class UnionFind:
        def __init__(self, items: Iterable[str]):
            self.parent = {item: item for item in items}

        def find(self, item: str) -> str:
            while self.parent[item] != item:
                self.parent[item] = self.parent[
                    self.parent[item]
                ]
                item = self.parent[item]
            return item

        def union(self, left: str, right: str):
            left_root = self.find(left)
            right_root = self.find(right)
            if left_root != right_root:
                self.parent[right_root] = left_root


    def make_log(
        severity,
        log_type,
        source_row_id,
        product_id,
        product_name,
        formulation_id,
        detail,
        raw_excerpt="",
    ):
        return {
            "severity": severity,
            "log_type": log_type,
            "source_row_id": source_row_id,
            "product_id": product_id,
            "product_name": product_name,
            "formulation_id": formulation_id,
            "detail": detail,
            "raw_excerpt": normalize_spaces(raw_excerpt)[:800],
        }


    def formula_string_for_group(group: pd.DataFrame) -> str:
        group = group.sort_values("ingredient_order")
        component_count = group["component_index"].nunique()

        if component_count <= 1:
            return "|".join(
                group["canonical_name"].astype(str)
            )

        return "|".join(
            f"C{row.component_index}::{row.canonical_name}"
            for row in group.itertuples()
        )


In [ ]:
if RUN_PREPROCESSING:
    product_id_col = PRODUCT_ID_COL
    product_name_col = PRODUCT_NAME_COL
    ingredient_text_col = INGREDIENT_TEXT_COL
    master_df = mfds_master_df
    manual_aliases = MANUAL_ALIASES
    extra_ingredient_lexicon = EXTRA_INGREDIENT_LEXICON
    name_similarity_threshold = NAME_SIMILARITY_THRESHOLD
    whitespace_min_coverage = WHITESPACE_MIN_COVERAGE


In [ ]:
if RUN_PREPROCESSING:
    logs = []

    master_loaded = (
        master_df is not None
        and not master_df.empty
    )

    if master_loaded:
        (
            canonical_map,
            alias_map,
            manual_map,
            metadata_map,
            ambiguous_alias_map,
        ) = build_ingredient_maps(
            master_df,
            manual_aliases,
        )
    else:
        canonical_map = {}
        alias_map = {}
        manual_map = {
            comparison_key(raw): canonical
            for raw, canonical in manual_aliases.items()
        }
        metadata_map = {}
        ambiguous_alias_map = {}

    # ------------------------------------------------------------------
    # Pass 1: 대상 제형 선택 및 쉼표형 데이터로 Lexicon 보강
    # ------------------------------------------------------------------
    formulation_records = []
    lexicon = set(extra_ingredient_lexicon)

    if master_loaded:
        lexicon.update(
            master_df["standard_name"]
            .dropna()
            .astype(str)
        )
        for aliases in master_df.get(
            "aliases",
            pd.Series(dtype=str),
        ).dropna():
            lexicon.update(split_alias_names(aliases))

    lexicon.update(manual_aliases.keys())
    lexicon.update(manual_aliases.values())


In [ ]:
if RUN_PREPROCESSING:
    for _, row in raw_df.iterrows():
        source_row_id = int(row["source_row_id"])
        product_id = normalize_spaces(row[product_id_col])
        product_name = normalize_spaces(row[product_name_col])
        ingredient_text = normalize_spaces(
            row[ingredient_text_col]
        )

        sections, meta = choose_target_sections(
            product_name,
            ingredient_text,
        )

        if meta["status"] != "OK":
            severity = (
                "ERROR"
                if meta["status"] in {
                    "EMPTY_OR_CRAWL_FAILED",
                    "NO_TARGET_SECTION",
                }
                else "WARNING"
            )
            logs.append(
                make_log(
                    severity,
                    meta["status"],
                    source_row_id,
                    product_id,
                    product_name,
                    "",
                    meta.get("reason", ""),
                    ingredient_text,
                )
            )
            continue

        formulation_id = f"{source_row_id:05d}_F01"

        for removed in meta.get(
            "removed_secondary_sections",
            [],
        ):
            logs.append(
                make_log(
                    "INFO",
                    "SECONDARY_PRODUCT_REMOVED",
                    source_row_id,
                    product_id,
                    product_name,
                    formulation_id,
                    "본품 뒤에 붙은 증정/부가 제형 성분을 제거했습니다.",
                    removed,
                )
            )

        # Components are merged into one formulation.
        for component_index, section in enumerate(
            sections,
            start=1,
        ):
            body = normalize_raw_ingredient_text(
                section.body
            )
            method = choose_tokenizer_method(body)
            tokens_first = (
                tokenize_delimited(body, method)
                if method != "WHITESPACE_LEXICON"
                else None
            )

            if tokens_first and not master_loaded:
                for token in tokens_first:
                    name = extract_concentration(
                        token
                    )["ingredient_name_clean"]
                    if name:
                        lexicon.add(name)

            formulation_records.append({
                "source_row_id": source_row_id,
                "product_id": product_id,
                "product_name_raw": product_name,
                "product_name_clean": clean_product_name(
                    product_name
                ),
                "formulation_id": formulation_id,
                "component_index": component_index,
                "component_name": section.label,
                "formulation_text": body,
                "tokenizer_method": method,
                "tokens_first_pass": tokens_first,
            })


In [ ]:
if RUN_PREPROCESSING:
    trie = build_ingredient_trie(lexicon)

    # ------------------------------------------------------------------
    # Pass 2: Long 변환 및 MFDS 표준명 매핑
    # ------------------------------------------------------------------
    long_rows = []
    order_counter = defaultdict(int)


In [ ]:
if RUN_PREPROCESSING:
    for record in formulation_records:
        tokens = record["tokens_first_pass"]
        coverage = 1.0
        unknown = []

        if record["tokenizer_method"] == "WHITESPACE_LEXICON":
            tokens, unknown, coverage = (
                segment_with_ingredient_trie(
                    record["formulation_text"],
                    trie,
                )
            )

            if (
                coverage < whitespace_min_coverage
                or unknown
            ):
                logs.append(
                    make_log(
                        "WARNING",
                        "LOW_WHITESPACE_PARSE_COVERAGE",
                        record["source_row_id"],
                        record["product_id"],
                        record["product_name_raw"],
                        record["formulation_id"],
                        (
                            f"coverage={coverage:.3f}; "
                            f"unresolved={' | '.join(unknown) or '없음'}"
                        ),
                        record["formulation_text"],
                    )
                )

        if tokens and record["tokenizer_method"] != "WHITESPACE_LEXICON":
            tokens, mixed_expanded_count = expand_mixed_delimited_tokens(
                tokens,
                trie,
            )
            if mixed_expanded_count:
                logs.append(
                    make_log(
                        "INFO",
                        "MIXED_TOKEN_SPLIT_BY_MFDS",
                        record["source_row_id"],
                        record["product_id"],
                        record["product_name_raw"],
                        record["formulation_id"],
                        f"공백으로 붙은 토큰 {mixed_expanded_count}개를 MFDS 사전으로 재분리",
                        record["formulation_text"],
                    )
                )

        if not tokens:
            logs.append(
                make_log(
                    "ERROR",
                    "NO_INGREDIENT_TOKEN",
                    record["source_row_id"],
                    record["product_id"],
                    record["product_name_raw"],
                    record["formulation_id"],
                    "성분 토큰 0개",
                    record["formulation_text"],
                )
            )
            continue

        for raw_token in tokens:
            parsed = extract_concentration(raw_token)
            clean_name = parsed["ingredient_name_clean"]

            if not clean_name:
                logs.append(
                    make_log(
                        "ERROR",
                        "EMPTY_INGREDIENT_TOKEN",
                        record["source_row_id"],
                        record["product_id"],
                        record["product_name_raw"],
                        record["formulation_id"],
                        "빈 성분명",
                        raw_token,
                    )
                )
                continue

            canonical = map_to_canonical(
                clean_name,
                canonical_map,
                alias_map,
                manual_map,
                metadata_map,
                ambiguous_alias_map,
                master_loaded,
            )

            if canonical.get("ambiguous_candidates"):
                logs.append(
                    make_log(
                        "WARNING",
                        "AMBIGUOUS_MFDS_ALIAS",
                        record["source_row_id"],
                        record["product_id"],
                        record["product_name_raw"],
                        record["formulation_id"],
                        (
                            f"{clean_name} -> "
                            f"{canonical['ambiguous_candidates']}"
                        ),
                        raw_token,
                    )
                )

            order_counter[record["formulation_id"]] += 1

            long_rows.append({
                "source_row_id": record["source_row_id"],
                "product_id": record["product_id"],
                "product_name_raw": record["product_name_raw"],
                "product_name_clean": record["product_name_clean"],
                "formulation_id": record["formulation_id"],
                "component_index": record["component_index"],
                "component_name": record["component_name"],
                "ingredient_order": order_counter[
                    record["formulation_id"]
                ],
                "ingredient_raw": normalize_spaces(raw_token),
                "ingredient_name_clean": clean_name,
                "canonical_name": canonical["canonical_name"],
                "ingredient_code": canonical["ingredient_code"],
                "canonical_english_name": canonical[
                    "canonical_english_name"
                ],
                "cas_no": canonical["cas_no"],
                "mapping_status": canonical["mapping_status"],
                "mapping_source": canonical["mapping_source"],
                "concentration_raw": parsed["concentration_raw"],
                "concentration_value": parsed[
                    "concentration_value"
                ],
                "concentration_unit_original": parsed[
                    "concentration_unit_original"
                ],
                "concentration_ppm": parsed[
                    "concentration_ppm"
                ],
                "has_declared_concentration": parsed[
                    "has_declared_concentration"
                ],
                "tokenizer_method": record[
                    "tokenizer_method"
                ],
                "parse_coverage_ratio": coverage,
            })


In [ ]:
if RUN_PREPROCESSING:
    long_df = pd.DataFrame(long_rows)

    if long_df.empty:
        raise RuntimeError("Long 데이터가 생성되지 않았습니다.")

    # ------------------------------------------------------------------
    # Formula Hash 및 동일 SKU 그룹화
    # ------------------------------------------------------------------
    formula_strings = (
        long_df
        .groupby("formulation_id", sort=False)
        .apply(
            formula_string_for_group,
            include_groups=False,
        )
    )
    formula_hash_map = formula_strings.map(
        sha256_text
    ).to_dict()
    long_df["formula_hash"] = long_df[
        "formulation_id"
    ].map(formula_hash_map)

    formulation_summary = (
        long_df[[
            "formulation_id",
            "source_row_id",
            "product_id",
            "product_name_raw",
            "product_name_clean",
            "formula_hash",
        ]]
        .drop_duplicates("formulation_id")
        .reset_index(drop=True)
    )

    product_group_map = {}


In [ ]:
if RUN_PREPROCESSING:
    for formula_hash, group in formulation_summary.groupby(
        "formula_hash"
    ):
        ids = group["formulation_id"].tolist()
        uf = UnionFind(ids)
        rows = group.to_dict("records")

        for left_index in range(len(rows)):
            for right_index in range(
                left_index + 1,
                len(rows),
            ):
                left = rows[left_index]
                right = rows[right_index]
                similarity = name_similarity(
                    left["product_name_clean"],
                    right["product_name_clean"],
                )
                same_id = (
                    left["product_id"]
                    == right["product_id"]
                )

                if (
                    similarity >= name_similarity_threshold
                    or same_id
                ):
                    uf.union(
                        left["formulation_id"],
                        right["formulation_id"],
                    )
                else:
                    logs.append(
                        make_log(
                            "INFO",
                            "SAME_FORMULA_DIFFERENT_NAME",
                            left["source_row_id"],
                            left["product_id"],
                            left["product_name_raw"],
                            left["formulation_id"],
                            (
                                f"상대={right['product_name_raw']} | "
                                f"유사도={similarity:.3f}"
                            ),
                            formula_hash[:24],
                        )
                    )

        clusters = defaultdict(list)
        for formulation_id in ids:
            clusters[uf.find(formulation_id)].append(
                formulation_id
            )

        for cluster_number, members in enumerate(
            sorted(
                clusters.values(),
                key=lambda values: sorted(values)[0],
            ),
            start=1,
        ):
            group_id = "PG_" + sha256_text(
                f"{formula_hash}|{cluster_number}|"
                f"{'|'.join(sorted(members))}"
            )[:16]

            for formulation_id in members:
                product_group_map[formulation_id] = group_id

    long_df["product_group_id"] = long_df[
        "formulation_id"
    ].map(product_group_map)

    summary_with_group = formulation_summary.copy()
    summary_with_group["product_group_id"] = (
        summary_with_group["formulation_id"]
        .map(product_group_map)
    )


In [ ]:
if RUN_PREPROCESSING:
    sku_counts = (
        summary_with_group
        .groupby("product_group_id")["product_id"]
        .nunique()
        .to_dict()
    )

    long_df["sku_count"] = (
        long_df["product_group_id"]
        .map(sku_counts)
        .astype(int)
    )
    long_df["duplicate_sku_flag"] = (
        long_df["sku_count"]
        .gt(1)
        .astype(int)
    )

    # ------------------------------------------------------------------
    # QA
    # ------------------------------------------------------------------
    suspicious_patterns = [
        "사용방법",
        "손상된피부",
        "효능효과",
        "창상피복재",
        "전성분정제수",
        "카밍패드",
        "pdrn앰플",
        "수분크림정제수",
        "※자연유래",
        "x5:ad5",
    ]


In [ ]:
if RUN_PREPROCESSING:
    for row in long_df.itertuples():
        name = str(row.ingredient_name_clean)
        name_key = comparison_key(name)

        if len(name) > 120:
            logs.append(
                make_log(
                    "WARNING",
                    "SUSPICIOUS_LONG_INGREDIENT_TOKEN",
                    row.source_row_id,
                    row.product_id,
                    row.product_name_raw,
                    row.formulation_id,
                    f"성분명 길이={len(name)}",
                    name,
                )
            )

        if (
            any(
                pattern in name_key
                for pattern in suspicious_patterns
            )
            or name.endswith("(")
            or name.startswith(("\"", "'"))
            or name.endswith(("\"", "'"))
            or re.search(r"\d(?:ppm|ppb|%)$", name, re.I)
        ):
            logs.append(
                make_log(
                    "WARNING",
                    "SUSPICIOUS_INGREDIENT_TEXT",
                    row.source_row_id,
                    row.product_id,
                    row.product_name_raw,
                    row.formulation_id,
                    f"의심 성분명={name}",
                    name,
                )
            )

        ppm = row.concentration_ppm
        if pd.notna(ppm) and float(ppm) > 1_000_000:
            logs.append(
                make_log(
                    "WARNING",
                    "CONCENTRATION_OVER_100_PERCENT",
                    row.source_row_id,
                    row.product_id,
                    row.product_name_raw,
                    row.formulation_id,
                    f"{name}: {float(ppm):,.3f} ppm",
                    row.ingredient_raw,
                )
            )

    ingredient_counts = long_df.groupby(
        "formulation_id"
    ).size()

    for formulation_id, count in ingredient_counts.items():
        if count < 3 or count > 250:
            sample = long_df[
                long_df["formulation_id"].eq(formulation_id)
            ].iloc[0]
            logs.append(
                make_log(
                    "WARNING",
                    "ABNORMAL_INGREDIENT_COUNT",
                    sample.source_row_id,
                    sample.product_id,
                    sample.product_name_raw,
                    formulation_id,
                    f"성분 수={int(count)}",
                    "",
                )
            )

    # 같은 component 안에서만 중복을 검사합니다.


In [ ]:
if RUN_PREPROCESSING:
    duplicate_rows = (
        long_df
        .groupby([
            "formulation_id",
            "component_index",
            "canonical_name",
        ])
        .size()
        .reset_index(name="count")
    )

    for row in duplicate_rows[
        duplicate_rows["count"] > 1
    ].itertuples():
        sample = long_df[
            long_df["formulation_id"].eq(
                row.formulation_id
            )
        ].iloc[0]
        logs.append(
            make_log(
                "INFO",
                "DUPLICATE_INGREDIENT_WITHIN_COMPONENT",
                sample.source_row_id,
                sample.product_id,
                sample.product_name_raw,
                row.formulation_id,
                f"{row.canonical_name} {row.count}회",
                "",
            )
        )

    # MFDS 미매칭은 고유 성분별 한 줄만 기록합니다.
    if master_loaded:
        unmatched = long_df[
            long_df["mapping_status"].eq(
                "UNMATCHED_MFDS"
            )
        ]

        for name, group in unmatched.groupby(
            "ingredient_name_clean"
        ):
            logs.append(
                make_log(
                    "WARNING",
                    "UNMATCHED_MFDS_INGREDIENT",
                    int(group["source_row_id"].iloc[0]),
                    group["product_id"].iloc[0],
                    group["product_name_raw"].iloc[0],
                    group["formulation_id"].iloc[0],
                    (
                        f"미매칭={name} | "
                        f"제품수={group['product_id'].nunique()}"
                    ),
                    name,
                )
            )


In [ ]:
if RUN_PREPROCESSING:
    for clean_name, group in formulation_summary.groupby(
        "product_name_clean"
    ):
        if (
            clean_name
            and group["formula_hash"].nunique() > 1
        ):
            sample = group.iloc[0]
            logs.append(
                make_log(
                    "WARNING",
                    "SAME_NAME_DIFFERENT_FORMULA",
                    sample.source_row_id,
                    sample.product_id,
                    sample.product_name_raw,
                    sample.formulation_id,
                    (
                        f"정제 상품명 '{clean_name}'에 "
                        f"{group['formula_hash'].nunique()}개 처방"
                    ),
                    "",
                )
            )

    # API 사전이 있는데도 매칭률이 낮으면 조용히 통과시키지 않습니다.
    if master_loaded:
        matched_statuses = {
            "EXACT_MFDS",
            "MFDS_ALIAS",
            "MFDS_PAREN_FALLBACK",
            "MANUAL_ALIAS",
        }
        match_rate = long_df["mapping_status"].isin(
            matched_statuses
        ).mean()

        if match_rate < 0.80:
            logs.append(
                make_log(
                    "ERROR",
                    "MFDS_MAPPING_RATE_TOO_LOW",
                    "",
                    "",
                    "",
                    "",
                    (
                        f"MFDS 매칭률={match_rate:.2%}. "
                        "API 캐시 또는 컬럼 연결을 확인하세요."
                    ),
                    "",
                )
            )

    log_df = pd.DataFrame(logs)

    if not log_df.empty:
        severity_order = {
            "ERROR": 0,
            "WARNING": 1,
            "INFO": 2,
        }
        log_df["_order"] = (
            log_df["severity"]
            .map(severity_order)
            .fillna(9)
        )
        log_df = (
            log_df
            .sort_values([
                "_order",
                "source_row_id",
                "log_type",
            ])
            .drop(columns="_order")
            .reset_index(drop=True)
        )


In [ ]:
if RUN_PREPROCESSING:
    final_columns = [
        "source_row_id",
        "product_id",
        "product_name_raw",
        "product_name_clean",
        "product_group_id",
        "duplicate_sku_flag",
        "sku_count",
        "formulation_id",
        "component_index",
        "component_name",
        "formula_hash",
        "ingredient_order",
        "ingredient_raw",
        "ingredient_name_clean",
        "canonical_name",
        "ingredient_code",
        "canonical_english_name",
        "cas_no",
        "mapping_status",
        "mapping_source",
        "concentration_raw",
        "concentration_value",
        "concentration_unit_original",
        "concentration_ppm",
        "has_declared_concentration",
        "tokenizer_method",
        "parse_coverage_ratio",
    ]

    long_df = (
        long_df[final_columns]
        .sort_values([
            "source_row_id",
            "ingredient_order",
        ])
        .reset_index(drop=True)
    )



## 셀 13. 핵심 Parser Unit Test

실제 오류가 발생했던 패턴을 코드 실행 전에 점검합니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    # 1) 1,2-헥산다이올과 함량 쉼표
    sample_text = (
        "정제수, 스쿠알란(150,000ppm), 글리세린, "
        "1,2-헥산다이올, "
        "부틸렌글라이콜다이카프릴레이트/다이카프레이트"
    )
    sample_tokens = safe_split_commas(sample_text)
    assert sample_tokens == [
        "정제수",
        "스쿠알란(150,000ppm)",
        "글리세린",
        "1,2-헥산다이올",
        "부틸렌글라이콜다이카프릴레이트/다이카프레이트",
    ]

    # 2) 쉼표가 화학 위치번호에만 있으면 공백형
    assert choose_tokenizer_method(
        "정제수 글리세린 1,2-헥산다이올 판테놀"
    ) == "WHITESPACE_LEXICON"

    # 3) 함량 환산 및 괄호 없는 함량
    assert extract_concentration(
        "스쿠알란(150,000ppm)"
    )["concentration_ppm"] == 150000
    assert extract_concentration(
        "판테놀(0.1%)"
    )["concentration_ppm"] == 1000
    assert extract_concentration(
        "병풀추출물(1000ppb)"
    )["concentration_ppm"] == 1
    assert extract_concentration(
        "병풀잎추출물0.2%"
    )["concentration_ppm"] == 2000
    assert extract_concentration(
        "병풀잎추출물0.2%"
    )["ingredient_name_clean"] == "병풀잎추출물"

    # 4) 따옴표·마침표 제거
    assert clean_ingredient_token('"정제수') == "정제수"
    assert clean_ingredient_token("클로페네신.") == "클로페네신"

    # 5) 설명문 절단
    normalized_description = normalize_raw_ingredient_text(
        "성분 : 정제수, 글리세린, 베타글루칸 "
        "손상된 피부 보호를 위해 사용합니다."
    )
    assert "손상된피부" not in comparison_key(
        normalized_description
    )
    assert "자연유래" not in comparison_key(
        normalize_raw_ingredient_text(
            "정제수, 리모넨※ ※자연유래 에센셜 오일 성분"
        )
    )

    # 6) 본품 뒤 부가 제형 절단


In [ ]:
if RUN_PREPROCESSING:
    main_text, removed_text = strip_inline_secondary_product(
        "정제수, 글리세린, 부틸렌글라이콜, 판테놀, 세라마이드이오피 "
        "PDRN 앰플 정제수, 글리세린, 나이아신아마이드"
    )
    assert "앰플" not in main_text
    assert "앰플" in removed_text

    # 7) 제품명 접두부 제거
    assert strip_leading_product_label(
        "PDRN 수분크림 정제수, 글리세린"
    ).startswith("정제수")

    # 8) 다중 선택상품 제외
    choice_flag, _ = is_choice_product(
        "수딩크림/젤크림 2종 택1"
    )
    assert choice_flag

    print("핵심 Unit Test 통과")
    display(sample_tokens)


핵심 Unit Test 통과


['정제수', '스쿠알란(150,000ppm)', '글리세린', '1,2-헥산다이올', '부틸렌글라이콜다이카프릴레이트/다이카프레이트']


## 셀 14. 전체 Raw → Long 변환 실행

API 사전이 준비되어 있으면 MFDS 표준명·이명을 이용합니다.  
API 사용을 끈 경우에도 기본 전처리는 실행되지만 `mapping_status`가 `NORMALIZED_ONLY`로 남습니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    if long_df.empty:
        raise RuntimeError(
            "Long 데이터가 생성되지 않았습니다. "
            "03_ambiguous_log 내용을 확인하세요."
        )

    MFDS_MATCHED_STATUSES = {
        "EXACT_MFDS",
        "MFDS_ALIAS",
        "MFDS_PAREN_FALLBACK",
        "MANUAL_ALIAS",
    }

    mfds_match_rate = long_df["mapping_status"].isin(
        MFDS_MATCHED_STATUSES
    ).mean()

    if USE_MFDS_API and mfds_match_rate < MIN_MFDS_MATCH_RATE:
        raise RuntimeError(
            f"MFDS 매칭률이 {mfds_match_rate:.2%}로 너무 낮습니다. "
            "API 표준 캐시와 컬럼 연결을 확인하세요. "
            "이 오류는 NORMALIZED_ONLY 결과가 최종본으로 저장되는 것을 막기 위한 것입니다."
        )

    if not KEEP_INFO_LOGS and not log_df.empty:
        log_df = log_df[
            log_df["severity"].isin(["ERROR", "WARNING"])
        ].reset_index(drop=True)

    print(f"Long 성분 행 수: {len(long_df):,}")
    print("변환된 원본 상품 행 수:", long_df["source_row_id"].nunique())
    print("고유 제품 그룹 수:", long_df["product_group_id"].nunique())
    print("고유 표준화 성분 수:", long_df["canonical_name"].nunique())
    print(f"MFDS/승인 Alias 매칭률: {mfds_match_rate:.2%}")

    display(long_df.head(10))


Long 성분 행 수: 10,329
변환된 원본 상품 행 수: 231
고유 제품 그룹 수: 211
고유 표준화 성분 수: 1292
MFDS/승인 Alias 매칭률: 99.64%


,source_row_id,product_id,product_name_raw,product_name_clean,product_group_id,duplicate_sku_flag,sku_count,formulation_id,component_index,component_name,formula_hash,ingredient_order,ingredient_raw,ingredient_name_clean,canonical_name,ingredient_code,canonical_english_name,cas_no,mapping_status,mapping_source,concentration_raw,concentration_value,concentration_unit_original,concentration_ppm,has_declared_concentration,tokenizer_method,parse_coverage_ratio
0,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,1,정제수,정제수,정제수,,"Water,Aqua",,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
1,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,2,글리세린,글리세린,글리세린,,Glycerin,,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
2,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,3,프로판다이올,프로판다이올,프로판다이올,,Propanediol,,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
3,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,4,카프릴릭/카프릭트라이글리세라이드,카프릴릭/카프릭트라이글리세라이드,카프릴릭/카프릭트라이글리세라이드,,Caprylic/Capric Triglyceride,,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
4,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,5,솔비탄스테아레이트,솔비탄스테아레이트,솔비탄스테아레이트,,Sorbitan Stearate,,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
5,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,6,세테아릴알코올,세테아릴알코올,세테아릴알코올,,Cetearyl Alcohol,"8005-44-5 ,67762-27-0",EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
6,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,7,잇꽃씨오일,잇꽃씨오일,잇꽃씨오일,,Carthamus Tinctorius (Safflower) Seed Oil,,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
7,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,8,글리세릴스테아레이트,글리세릴스테아레이트,글리세릴스테아레이트,,Glyceryl Stearate,,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
8,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,9,스테아릭애씨드,스테아릭애씨드,스테아릭애씨드,,Stearic Acid,,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0
9,0,A000000260257,[7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml),제로이드 수딩 크림,PG_bc5f1e8050cc5e5a,1,2,00000_F01,1,PRIMARY,45af8a8c199e9359420c5e0f65368d399ab8173b6cfd30a488eb9e2cf17f8ea2,10,다이메티콘,다이메티콘,다이메티콘,,Dimethicone,,EXACT_MFDS,MFDS,,NaN,,NaN,0,SAFE_COMMA,1.0



## 셀 15. 메인 출력 3개 저장

- 원본은 이미 저장되었으며 다시 한 번 동일 경로에 저장
- Long 변환본 저장
- 애매 로그 저장


In [ ]:
if RUN_PREPROCESSING:
    pass
    raw_df.to_csv(
        RAW_OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    long_df.to_csv(
        LONG_OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    log_df.to_csv(
        LOG_OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print("저장 완료")
    print("1.", RAW_OUTPUT_PATH)
    print("2.", LONG_OUTPUT_PATH)
    print("3.", LOG_OUTPUT_PATH)



## 셀 16. 결과 요약 및 품질 확인

우선 확인할 로그:

- `EMPTY_OR_CRAWL_FAILED`: 원본 재수집 필요
- `SKIPPED_MULTI_OPTION`: 의도적으로 제외한 다중 선택상품
- `UNMATCHED_MFDS_INGREDIENT`: API 표준명·이명 미매칭
- `LOW_WHITESPACE_PARSE_COVERAGE`: 공백형 성분 일부 미해결
- `SUSPICIOUS_LONG_INGREDIENT_TOKEN`: 성분 여러 개가 하나로 합쳐졌을 가능성
- `SUSPICIOUS_INGREDIENT_TEXT`: 설명문 또는 깨진 괄호 가능성

`INFO` 로그는 실제 실패가 아니라 동일 처방 후보나 원본 중복 성분 안내입니다.


In [ ]:
if RUN_PREPROCESSING:
    pass
    matched_statuses = {
        "EXACT_MFDS",
        "MFDS_ALIAS",
        "MFDS_PAREN_FALLBACK",
        "MANUAL_ALIAS",
    }

    summary = pd.DataFrame({
        "항목": [
            "원본 상품 행",
            "변환된 상품 행",
            "의도적 제외/실패 상품 행",
            "Long 성분 행",
            "고유 제품 그룹",
            "고유 표준화 성분",
            "함량 명시 성분 행",
            "MFDS 정확 매칭",
            "MFDS 이명 매칭",
            "MFDS 괄호 보정 매칭",
            "수동 Alias 매칭",
            "MFDS 미매칭",
            "MFDS 전체 매칭률",
            "ERROR 로그",
            "WARNING 로그",
            "INFO 로그",
        ],
        "값": [
            len(raw_df),
            long_df["source_row_id"].nunique(),
            len(raw_df) - long_df["source_row_id"].nunique(),
            len(long_df),
            long_df["product_group_id"].nunique(),
            long_df["canonical_name"].nunique(),
            int(long_df["has_declared_concentration"].sum()),
            int(long_df["mapping_status"].eq("EXACT_MFDS").sum()),
            int(long_df["mapping_status"].eq("MFDS_ALIAS").sum()),
            int(long_df["mapping_status"].eq("MFDS_PAREN_FALLBACK").sum()),
            int(long_df["mapping_status"].eq("MANUAL_ALIAS").sum()),
            int(long_df["mapping_status"].eq("UNMATCHED_MFDS").sum()),
            f"{long_df['mapping_status'].isin(matched_statuses).mean():.2%}",
            int(log_df["severity"].eq("ERROR").sum()) if not log_df.empty else 0,
            int(log_df["severity"].eq("WARNING").sum()) if not log_df.empty else 0,
            int(log_df["severity"].eq("INFO").sum()) if not log_df.empty else 0,
        ],
    })

    display(summary)

    print("\n로그 유형별 건수")
    if log_df.empty:
        print("로그 없음")
    else:
        display(
            log_df["log_type"]
            .value_counts()
            .rename_axis("log_type")
            .reset_index(name="count")
        )


In [ ]:
if RUN_PREPROCESSING:
    print("\nTokenizer별 상품 수")
    display(
        long_df[["source_row_id", "tokenizer_method"]]
        .drop_duplicates()["tokenizer_method"]
        .value_counts()
        .rename_axis("tokenizer_method")
        .reset_index(name="product_count")
    )

    print("\n다중 선택상품 제외 목록")
    if not log_df.empty:
        display(
            log_df[
                log_df["log_type"].isin([
                    "SKIPPED_MULTI_OPTION",
                    "SKIPPED_MULTIPLE_TARGET_PRODUCTS",
                ])
            ][[
                "source_row_id",
                "product_id",
                "product_name",
                "detail",
            ]]
        )


,항목,값
0,원본 상품 행,240
1,변환된 상품 행,231
2,의도적 제외/실패 상품 행,9
3,Long 성분 행,10329
4,고유 제품 그룹,211
5,고유 표준화 성분,1292
6,함량 명시 성분 행,301
7,MFDS 정확 매칭,10224
8,MFDS 이명 매칭,28
9,MFDS 괄호 보정 매칭,15



로그 유형별 건수


,log_type,count
0,UNMATCHED_MFDS_INGREDIENT,31
1,SAME_FORMULA_DIFFERENT_NAME,21
2,DUPLICATE_INGREDIENT_WITHIN_COMPONENT,10
3,EMPTY_OR_CRAWL_FAILED,6
4,MIXED_TOKEN_SPLIT_BY_MFDS,3
5,SKIPPED_MULTI_OPTION,3
6,AMBIGUOUS_MFDS_ALIAS,1
7,SECONDARY_PRODUCT_REMOVED,1



Tokenizer별 상품 수


,tokenizer_method,product_count
0,SAFE_COMMA,223
1,WHITESPACE_LEXICON,5
2,AT_DELIMITER,3



다중 선택상품 제외 목록


,source_row_id,product_id,product_name,detail
7,24,A000000211213,[수분보습폭탄] 빌리프X바나나킥 모이스춰라이징 밤 30ml 1+1 기획 (+키링파우치) / 아쿠아 밤 30ml 기획,슬래시로 구분된 복수 크림류 상품
27,124,A000000225277,[7월올영픽] 키엘 크림 기획/단품 (울트라 훼이셜 수분크림/메디크림/베리어크림),슬래시로 구분된 복수 크림류 상품
30,145,A000000229284,랑콤 이드라젠 모이스처라이징 수딩크림 / 젤크림 기획/단품 2종 택1 (50ml/30ml),선택상품 키워드: 택1



## 셀 17. 이번 Raw에서 문제가 있었던 대표 유형 재검수

특정 행 번호에 의존하지 않고 상품명 키워드로 결과를 확인합니다.

- 블랙스네일: 리뉴얼 적용 처방만
- 포들 캡슐크림: 젤+캡슐 병합
- 시카페어 3종세트: 메인 크림만
- 크림 토너 증정: 토너 제외
- 2종 택1: Long에 포함되지 않음


In [ ]:
if RUN_PREPROCESSING:
    pass
    CHECK_KEYWORDS = [
        "블랙스네일",
        "포들 밤 레티날",
        "히알루 B5",
        "에스네이처 아쿠아",
        "메디큐브 PDRN 핑크",
        "프로바이오덤 3D리프팅",
        "멜라-셋",
        "더모이스처 배리어",
        "마데카 크림 타임리버스",
        "제로이드 인텐시브",
        "랑콤 이드라젠",
    ]

    check_rows = []


In [ ]:
if RUN_PREPROCESSING:
    for keyword in CHECK_KEYWORDS:
        raw_match = raw_df[
            raw_df[PRODUCT_NAME_COL]
            .astype(str)
            .str.contains(keyword, na=False)
        ]

        for row in raw_match.itertuples():
            source_id = getattr(row, "source_row_id")
            long_match = long_df[
                long_df["source_row_id"].eq(source_id)
            ]
            log_match = (
                log_df[
                    log_df["source_row_id"].astype(str).eq(str(source_id))
                ]
                if not log_df.empty
                else pd.DataFrame()
            )

            suspicious = long_match[
                long_match["ingredient_name_clean"]
                .astype(str)
                .str.contains(
                    r"카밍패드|pdrn앰플|수분크림정제수|전성분정제수|자연유래",
                    case=False,
                    regex=True,
                    na=False,
                )
            ]

            check_rows.append({
                "source_row_id": source_id,
                "상품명": getattr(row, PRODUCT_NAME_COL),
                "Long행수": len(long_match),
                "구성요소": (
                    " | ".join(
                        long_match["component_name"]
                        .drop_duplicates()
                        .astype(str)
                    )
                    if not long_match.empty else ""
                ),
                "MFDS매칭률": (
                    f"{long_match['mapping_status'].isin(matched_statuses).mean():.1%}"
                    if not long_match.empty else ""
                ),
                "의심토큰수": len(suspicious),
                "로그": (
                    " | ".join(
                        log_match["log_type"]
                        .drop_duplicates()
                        .astype(str)
                    )
                    if not log_match.empty else ""
                ),
            })

    display(pd.DataFrame(check_rows))

    # 최종 안전검사


In [ ]:
if RUN_PREPROCESSING:
    remaining_suspicious = long_df[
        long_df["ingredient_name_clean"]
        .astype(str)
        .str.contains(
            r"카밍패드|pdrn앰플|수분크림정제수|전성분정제수|자연유래|X5:AD5",
            case=False,
            regex=True,
            na=False,
        )
    ]

    print("최종 의심 토큰 수:", len(remaining_suspicious))
    display(remaining_suspicious.head(20))


,source_row_id,상품명,Long행수,구성요소,MFDS매칭률,의심토큰수,로그
0,30,[1+1/광채탄력] 닥터지 블랙스네일 크림 50ml 1+1 기획 +(15ml 추가 증정),74,리뉴얼 적용,100.0%,0,
1,103,[웬디PICK] 포들 밤 레티날 모공 리셋 솔루션 캡슐 크림 55g,51,젤 | 캡슐,100.0%,0,
2,64,[7월 올영픽/온라인추가증정] 라로슈포제 히알루 B5 수분탄력 크림 50ml 기획 (+3ml+히알루세럼10ml),46,히알루B5 크림,100.0%,0,
3,2,[1+1/1등 모공 수분천재크림] 에스네이처 아쿠아 스쿠알란 수분크림 60ml 더블기획,27,PRIMARY,100.0%,0,
4,11,[수분충전/쿨링진정] 에스네이처 아쿠아 오아시스 수분 젤크림 90ml 기획 (+카밍패드 2매),24,젤크림,100.0%,0,
5,161,[1등 모공 수분천재크림] 에스네이처 아쿠아 스쿠알란 수분크림 60ml 기획(60ml+30ml),27,PRIMARY,100.0%,0,
6,28,[여배우광채크림] 메디큐브 PDRN 핑크 콜라겐 캡슐크림 55g 기획 (+4.5g*2ea),59,PRIMARY,98.3%,0,UNMATCHED_MFDS_INGREDIENT
7,50,[수분진정/화잘먹] 메디큐브 PDRN 핑크 히알루로닉 수분크림 100ml 더블기획,58,PRIMARY,100.0%,0,
8,85,[속건조수분크림] 메디큐브 PDRN 핑크 히알루로닉 수분크림 50ml 더블기획 (+앰플10ml),57,PRIMARY,100.0%,0,SECONDARY_PRODUCT_REMOVED
9,44,[카디비PICK|탄력보습] 바이오힐보 프로바이오덤 3D리프팅 크림 50ml [단품/기획],79,PRIMARY,98.7%,0,UNMATCHED_MFDS_INGREDIENT


최종 의심 토큰 수: 0


,source_row_id,product_id,product_name_raw,product_name_clean,product_group_id,duplicate_sku_flag,sku_count,formulation_id,component_index,component_name,formula_hash,ingredient_order,ingredient_raw,ingredient_name_clean,canonical_name,ingredient_code,canonical_english_name,cas_no,mapping_status,mapping_source,concentration_raw,concentration_value,concentration_unit_original,concentration_ppm,has_declared_concentration,tokenizer_method,parse_coverage_ratio



## 다음 Feature Engineering 단계에서의 사용

이 Long 데이터에는 공통 성분을 포함한 모든 성분을 보존합니다.

이후 별도 코드에서 비교할 권장 실험:

1. 모든 성분 원핫
2. 출현율 98% 이상 성분 제거
3. 출현율 95% 이상 성분 제거
4. 출현율 90% 이상 성분 제거
5. 과거 방식처럼 출현율 상위 5개 제거
6. 기능군 Count·Ratio 추가
7. 주성분 Top 5·Top 10 추가
8. 제품명 Claim 및 성분 일치 여부 추가
9. 피부유형×기능군 Interaction 추가

원핫 생성 시 같은 성분이 원본에 두 번 있어도 값이 2가 되지 않도록 `aggfunc="max"`를 사용해야 합니다.


## 저장된 전처리 결과 미리보기

In [ ]:
import pandas as pd

ingredient_final = pd.read_csv(
    DATA_INTERIM_DIR / "전성분_표준화_최종.csv",
    encoding="utf-8-sig",
)
print("전처리 완료 데이터:", ingredient_final.shape)
display(ingredient_final.head())
